# Fábrica de Corpus CitizenTech
## Agricultura familiar: a quem eu tenho direito e em que porta eu bato

**Global Solution | Processamento de Linguagem Natural, Chatbots e Virtual Agents**
Prof. Dr. Sérgio Eduardo Palmiere | FIAP | Turma 2TIAPR

| Integrante | RM |
|---|---|
| Claudio Alves | 568565 |
| Gustavo Duarte | 567066 |
| Leonardo Pereira | 567836 |

---

Este notebook transforma documentos oficiais sobre agricultura familiar em três corpus
prontos para o fine-tuning de um SLM. Ele roda do começo ao fim sem intervenção: baixa as
fontes, extrai, limpa, normaliza, anonimiza, segmenta, gera os corpus, mede a qualidade e
sobe uma interface Gradio.

**Como rodar:** `Ambiente de execução > Executar tudo`. A primeira célula instala as
dependências. GPU é opcional e serve só para a seção 7.3, que é um experimento.

Além das oito etapas obrigatórias, incluímos quatro verificações que não estavam no
enunciado e que mudaram decisões de projeto: um portão de qualidade da extração, um teste
adversarial da anonimização, um verificador automático de fidelidade das respostas e uma
detecção de quase-duplicatas. Cada um está marcado como **passo nosso** e aparece com o
número que produziu.

## 1. O problema, a persona e o ODS

### Por que agricultura familiar, e por que um modelo local

O governo mantém várias políticas para a agricultura familiar: CAF, Pronaf, Garantia-Safra,
assistência técnica. As regras existem e estão publicadas. O problema é a distância entre o
documento e quem precisa dele.

Um detalhe define a escolha de arquitetura: no campo a internet é ruim o ano inteiro, não só
num dia de crise. Um assistente que depende de nuvem falha com frequência justamente onde
seria mais útil. Um SLM de poucos bilhões de parâmetros cabe num celular ou no computador do
sindicato e responde offline, desde que tenha sido treinado antes com o conteúdo certo. É
esse conteúdo que a GS produz.

### A persona

Dona Alzira, 54 anos. Três hectares, planta feijão e milho, duas das cinco culturas
cobertas pelo Garantia-Safra. Estudou até a quinta série. O técnico da Emater passa a cada
dois meses e o sinal de celular só pega perto da estrada.

As dúvidas dela são três:

1. **Eu me encaixo?** O CAF está ativo? A renda da família passou do limite? A aposentadoria
   do marido entra na conta?
2. **Perdi a lavoura na estiagem, tinha direito a alguma coisa?** Precisava estar inscrita
   antes, e ela não sabia.
3. **Em que porta eu bato?** Sindicato, Emater, banco, prefeitura, entidade emissora do CAF.
   São nomes diferentes para etapas diferentes, e ninguém explica a ordem.

A resposta está escrita e é pública, mas espalhada entre uma cartilha de 44 páginas, uma lei
federal, um manual feito para o gestor municipal e um modelo de contrato.

### O ODS

| ODS | Meta | Como o projeto se conecta |
|---|---|---|
| 2. Fome zero e agricultura sustentável | 2.3 dobrar a renda dos pequenos produtores, inclusive por meio de acesso a serviços financeiros | É a meta principal: aqui o gargalo é o acesso, não a falta de política |
| 2. Fome zero e agricultura sustentável | 2.4 produção resiliente, com adaptação a eventos climáticos extremos | É o que o Garantia-Safra cobre, e por isso os avisos do INMET entram no corpus |
| 1. Erradicação da pobreza | 1.4 acesso a recursos econômicos e serviços financeiros | O crédito rural para a agricultura familiar se encaixa direto |

### Três decisões que tomamos antes de começar

1. **Resposta sempre extraída do documento.** Resposta inventada sobre quem tem direito a
   benefício público é pior do que não responder.
2. **Canal de atendimento não é dado pessoal.** Um anonimizador comum apaga o 0800 e o
   e-mail da ouvidoria junto com os telefones pessoais, e aí some a porta que a persona
   precisa achar.
3. **O assistente precisa saber dizer "não sei, procure fulano".** Indicar agrotóxico,
   comparar banco ou calcular aposentadoria são coisas que ele não deve fazer, e o corpus
   ensina isso de forma explícita.

In [1]:
# @title Instalacao das dependencias
# Rode esta celula uma vez. No Colab leva cerca de 1 minuto.
import subprocess, sys, importlib

PACOTES = {
    "pypdf": "pypdf",                 # extracao de PDF
    "pdfplumber": "pdfplumber",       # extracao de PDF com layout (comparacao)
    "docx": "python-docx",            # extracao de DOCX
    "bs4": "beautifulsoup4",          # extracao de HTML
    "lxml": "lxml",                   # parser rapido para o bs4
    "transformers": "transformers",   # tokenizer real do SLM alvo
    "spacy": "spacy",                 # NER para nomes de pessoas
    "gradio": "gradio",               # interface
}

faltando = [pip for mod, pip in PACOTES.items() if importlib.util.find_spec(mod) is None]
if faltando:
    print("Instalando:", ", ".join(faltando))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltando], check=False)
else:
    print("Todas as bibliotecas ja estao instaladas.")

if importlib.util.find_spec("pt_core_news_sm") is None:
    print("Baixando modelo de portugues do spaCy (pt_core_news_sm)...")
    subprocess.run([sys.executable, "-m", "spacy", "download", "pt_core_news_sm", "-q"], check=False)

print("Setup concluido.")

Todas as bibliotecas ja estao instaladas.
Setup concluido.


In [2]:
# @title Imports, configuracao e pastas de trabalho
import hashlib
import json
import os
import random
import re
import textwrap
import unicodedata
from collections import Counter, defaultdict

SEED = 42
random.seed(SEED)

PASTA_FONTES = "fontes"
PASTA_CORPUS = "corpus"
os.makedirs(PASTA_FONTES, exist_ok=True)
os.makedirs(PASTA_CORPUS, exist_ok=True)

# Tokens por trecho, conforme o enunciado (etapa 6).
TOKENS_MIN, TOKENS_MAX = 256, 512

# Modelo alvo do fine-tuning. Usamos o tokenizer DELE para contar tokens,
# em vez de contar palavras: o que vale e o que o modelo vai enxergar.
SLM_ALVO = "Qwen/Qwen2.5-0.5B-Instruct"


def titulo(txt, c="="):
    print("\n" + c * 78)
    print(txt)
    print(c * 78)


def tabela(linhas, cabecalho, larguras):
    """Imprime uma tabela simples e alinhada, truncando o que nao couber."""
    fmt = "  ".join("{:<" + str(w) + "}" for w in larguras)
    print(fmt.format(*cabecalho))
    print(fmt.format(*["-" * w for w in larguras]))
    for linha in linhas:
        campos = [str(c) if len(str(c)) <= w else str(c)[: w - 1] + "." for c, w in zip(linha, larguras)]
        print(fmt.format(*campos))


def amostra(txt, n=400):
    return textwrap.fill(re.sub(r"\s+", " ", txt)[:n], 100)


print(f"Seed fixa: {SEED}")
print(f"Alvo de segmentacao: {TOKENS_MIN} a {TOKENS_MAX} tokens")
print(f"Tokenizer de referencia: {SLM_ALVO}")

Seed fixa: 42
Alvo de segmentacao: 256 a 512 tokens
Tokenizer de referencia: Qwen/Qwen2.5-0.5B-Instruct


## 2. Ingestão e extração

O enunciado pede PDF, TXT, DOCX, HTML e texto colado. Em vez de converter tudo para um
formato só e fingir que atendemos, fomos atrás de documentos que **já existem** em cada
formato, o que também deixou o corpus mais variado:

| Formato | Documento | Órgão |
|---|---|---|
| PDF | Como acessar Crédito Rural pelo Pronaf: passo a passo | MDA |
| PDF | Orientações do Programa Garantia-Safra | Ministério da Agricultura |
| DOCX | Modelo de contrato de prestação de serviços de ATER | MDA |
| HTML | Lei 10.420/2002, que criou o Garantia-Safra | Planalto |
| TXT | Avisos meteorológicos oficiais | INMET |
| colado | Recado sobre a adesão ao Garantia-Safra | escrito por nós, para testar o canal |

São cinco documentos, dentro da faixa de 3 a 5 que o enunciado pede, mais o canal de texto
colado, que ele cobra à parte. Os PDFs somam 56 páginas, acima das 20 exigidas.

**Duas escolhas que vale explicar.**

A cartilha do Pronaf é distribuída pelo link que a própria notícia do MDA publica, e o
rodapé da página diz que a reprodução é gratuita desde que citada a fonte. Descartamos
várias cartilhas boas da Embrapa justamente por trazerem "todos os direitos reservados": o
enunciado pede fontes públicas, e isso tem que valer na prática, não só na intenção.

O boletim do INMET entra porque o gatilho do Garantia-Safra é climático: o benefício existe
para quem perde lavoura por seca ou por excesso de chuva. Além de cobrir o formato TXT, ele
traz o vocabulário do aviso oficial, que é o que chega ao produtor antes da perda.

Cada trecho extraído carrega de onde veio, arquivo e página ou seção, desde o primeiro
passo. Sem isso não há como rastrear uma resposta até a fonte lá no fim.

In [3]:
# @title Etapa 1 - Ingestao: baixar as fontes oficiais
# Baixamos dentro do proprio notebook para que qualquer pessoa reproduza a GS do zero.
# Se preferir, suba os arquivos a mao para a pasta "fontes/": o resto roda igual.
import json
import time
import urllib.request

FONTES = [
    {"arquivo": "cartilha_pronaf_credito_rural.pdf", "formato": "PDF",
     "orgao": "Ministerio do Desenvolvimento Agrario e Agricultura Familiar (MDA)",
     "titulo": "Como acessar Credito Rural pelo Pronaf: passo a passo",
     "url": "https://www.conexsus.org/validacao/wp-content/uploads/2024/11/cartilha-pronaf-final.pdf"},
    {"arquivo": "manual_garantia_safra_MAPA.pdf", "formato": "PDF",
     "orgao": "Ministerio da Agricultura - Departamento de Gestao de Riscos",
     "titulo": "Orientacoes do Programa Garantia-Safra",
     "url": "https://www.gov.br/mda/pt-br/acesso-a-informacao/acoes-e-programas/programas-projetos-acoes-obras-e-atividades/programa-garantia-safra/arquivos/manual6_atualizado_julho2019_orient-doc.pdf"},
    {"arquivo": "modelo_contrato_ater_MDA.docx", "formato": "DOCX",
     "orgao": "MDA - Departamento de Credito e Assistencia Tecnica",
     "titulo": "Modelo de contrato de prestacao de servicos de ATER",
     "url": "https://www.gov.br/mda/pt-br/acesso-a-informacao/copy_of_acoes-e-programas/credito/manuais-e-formularios/modelo-contrato-de-ater-decred-13-04.21"},
    {"arquivo": "lei_10420_2002_garantia_safra.html", "formato": "HTML",
     "orgao": "Presidencia da Republica (Planalto)",
     "titulo": "Lei 10.420/2002 - Fundo e Programa Garantia-Safra",
     "url": "https://www.planalto.gov.br/ccivil_03/leis/2002/l10420.htm",
     "espelho": "https://normas.leg.br/?urn=urn:lex:br:federal:lei:2002-04-10;10420"},
    {"arquivo": "avisos_oficiais_inmet.txt", "formato": "TXT",
     "orgao": "Instituto Nacional de Meteorologia (INMET)",
     "titulo": "Avisos meteorologicos oficiais (seca, geada, chuva excessiva)",
     "url": "https://apiprevmet3.inmet.gov.br/avisos/ativos"},
]

# A cartilha do Pronaf e distribuida pelo link que a propria noticia do MDA publica.
# A pagina do ministerio diz, no rodape: "A reproducao e gratuita desde que citada a
# fonte". Registramos isso aqui porque descartamos varias cartilhas boas da Embrapa
# justamente por trazerem "todos os direitos reservados".
NOTICIA_MDA_PRONAF = ("https://www.gov.br/mda/pt-br/noticias/2024/10/"
                      "cartilha-201ccomo-acessar-credito-rural-pelo-pronaf-passo-a-passo-"
                      "para-a-agricultura-familiar201d-e-lancada-em-brasilia")


def baixar(url, tentativas=12, espera=5, timeout=70):
    """O Planalto costuma derrubar a primeira conexao, entao insistimos."""
    for i in range(tentativas):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0", "Accept": "*/*"})
            return urllib.request.urlopen(req, timeout=timeout).read()
        except Exception as e:
            if i == tentativas - 1:
                raise
            print(f"    tentativa {i + 1} falhou ({type(e).__name__}), repetindo...")
            time.sleep(espera * (1 + i // 3))   # espera crescente


def boletim_inmet(bruto):
    """O INMET entrega JSON; o que interessa e o texto que o produtor le no aviso.
    Para quem planta, aviso de seca, geada ou chuva excessiva nao e curiosidade:
    e o gatilho do Garantia-Safra."""
    d = json.loads(bruto)
    linhas = ["AVISOS METEOROLOGICOS OFICIAIS",
              "Instituto Nacional de Meteorologia (INMET)",
              f"Coletado em {time.strftime('%d/%m/%Y')}", ""]
    for a in list(d.get("hoje", [])) + list(d.get("futuro", [])):
        linhas += [f"AVISO: {a.get('descricao', '')} - Severidade: {a.get('severidade', '')}",
                   f"Vigencia: de {a.get('inicio', '')} ate {a.get('fim', '')}"]
        if str(a.get("estados", "")).strip():
            linhas.append(f"Estados sob aviso: {a['estados']}")
        linhas += [f"Riscos previstos: {str(r).strip()}" for r in (a.get("riscos") or [])]
        linhas += [f"Orientacao a populacao: {str(i).strip()}" for i in (a.get("instrucoes") or [])]
        linhas.append("")
    return "\n".join(linhas)


for _f in FONTES:
    _destino = os.path.join(PASTA_FONTES, _f["arquivo"])
    if os.path.exists(_destino) and os.path.getsize(_destino) > 1024:
        print(f"[ja existe] {_f['arquivo']}")
        continue
    print(f"[{_f['formato']}] baixando {_f['arquivo']}")
    try:
        _bruto = baixar(_f["url"])
    except Exception:
        # O Planalto derruba a conexao com frequencia. Quando as tentativas acabam,
        # caimos no espelho oficial do LexML em vez de parar o notebook inteiro.
        if not _f.get("espelho"):
            raise
        print(f"    usando o espelho: {_f['espelho']}")
        _bruto = baixar(_f["espelho"])
    if _f["formato"] == "TXT":
        open(_destino, "w", encoding="utf-8").write(boletim_inmet(_bruto))
    else:
        open(_destino, "wb").write(_bruto)

json.dump(FONTES, open(os.path.join(PASTA_FONTES, "FONTES.json"), "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)

titulo("Fontes oficiais usadas")
tabela([(f["formato"], f["orgao"], f["titulo"],
         f"{os.path.getsize(os.path.join(PASTA_FONTES, f['arquivo'])) / 1024:.0f} KB") for f in FONTES],
       ["Formato", "Orgao", "Documento", "Tamanho"], [7, 40, 46, 8])

[ja existe] cartilha_pronaf_credito_rural.pdf
[ja existe] manual_garantia_safra_MAPA.pdf
[ja existe] modelo_contrato_ater_MDA.docx
[ja existe] lei_10420_2002_garantia_safra.html
[ja existe] avisos_oficiais_inmet.txt

Fontes oficiais usadas
Formato  Orgao                                     Documento                                       Tamanho 
-------  ----------------------------------------  ----------------------------------------------  --------
PDF      Ministerio do Desenvolvimento Agrario e.  Como acessar Credito Rural pelo Pronaf: passo.  4495 KB 
PDF      Ministerio da Agricultura - Departament.  Orientacoes do Programa Garantia-Safra          262 KB  
DOCX     MDA - Departamento de Credito e Assiste.  Modelo de contrato de prestacao de servicos d.  43 KB   
HTML     Presidencia da Republica (Planalto)       Lei 10.420/2002 - Fundo e Programa Garantia-S.  60 KB   
TXT      Instituto Nacional de Meteorologia (INM.  Avisos meteorologicos oficiais (seca, geada, .  6 KB    


In [4]:
# @title Etapa 2 - Extratores: um por formato, todos devolvendo texto + proveniencia
# Cada extrator devolve uma lista de blocos. Um bloco e sempre:
#   {"texto": ..., "fonte": <arquivo>, "local": <pagina/secao>}
# Guardar "local" desde a extracao e o que permite, la na frente, dizer de que
# pagina de que documento saiu cada trecho do corpus.


def extrair_pdf(caminho, motor="pypdf"):
    """PDF. Dois motores porque eles erram de formas diferentes (comparamos adiante)."""
    nome = os.path.basename(caminho)
    blocos = []
    if motor == "pypdf":
        from pypdf import PdfReader
        for i, pag in enumerate(PdfReader(caminho).pages, 1):
            blocos.append({"texto": pag.extract_text() or "", "fonte": nome, "local": f"p.{i}"})
    else:
        import pdfplumber
        with pdfplumber.open(caminho) as pdf:
            for i, pag in enumerate(pdf.pages, 1):
                blocos.append({"texto": pag.extract_text() or "", "fonte": nome, "local": f"p.{i}"})
    return blocos


def extrair_docx(caminho):
    """DOCX. Pegamos paragrafos e tambem o texto das tabelas, que o python-docx separa."""
    import docx
    nome = os.path.basename(caminho)
    doc = docx.Document(caminho)
    blocos, secao = [], "inicio"
    for p in doc.paragraphs:
        t = p.text.strip()
        if not t:
            continue
        if p.style.name.lower().startswith("heading"):
            secao = t[:60]
        blocos.append({"texto": t, "fonte": nome, "local": secao})
    for n, tab in enumerate(doc.tables, 1):
        celulas = [c.text.strip() for lin in tab.rows for c in lin.cells if c.text.strip()]
        if celulas:
            blocos.append({"texto": " | ".join(dict.fromkeys(celulas)), "fonte": nome, "local": f"tabela {n}"})
    return blocos


def extrair_html(caminho):
    """HTML. Tiramos script/style/nav e usamos os <h*> como marcador de secao."""
    from bs4 import BeautifulSoup
    nome = os.path.basename(caminho)
    bruto = open(caminho, "rb").read()
    for enc in ("utf-8", "latin-1", "cp1252"):
        try:
            html = bruto.decode(enc)
            break
        except UnicodeDecodeError:
            continue
    sopa = BeautifulSoup(html, "lxml")
    for tag in sopa(["script", "style", "nav", "header", "footer", "noscript"]):
        tag.decompose()
    blocos, secao = [], "inicio"
    for el in sopa.find_all(["h1", "h2", "h3", "p", "li", "td"]):
        t = el.get_text(" ", strip=True)
        if not t:
            continue
        if el.name in ("h1", "h2", "h3"):
            secao = t[:60]
            continue
        blocos.append({"texto": t, "fonte": nome, "local": secao})
    return blocos


def extrair_txt(caminho):
    """TXT. Quebramos em paragrafos (linha em branco) para nao virar um bloco unico."""
    nome = os.path.basename(caminho)
    texto = open(caminho, encoding="utf-8", errors="replace").read()
    partes = [p.strip() for p in re.split(r"\n\s*\n", texto) if p.strip()]
    return [{"texto": p, "fonte": nome, "local": f"bloco {i}"} for i, p in enumerate(partes, 1)]


def extrair_colado(texto, rotulo="texto_colado"):
    """Texto colado direto na ferramenta: mesmo contrato dos arquivos."""
    partes = [p.strip() for p in re.split(r"\n\s*\n", texto) if p.strip()]
    return [{"texto": p, "fonte": rotulo, "local": f"bloco {i}"} for i, p in enumerate(partes, 1)]


EXTRATORES = {".pdf": extrair_pdf, ".docx": extrair_docx,
              ".html": extrair_html, ".htm": extrair_html, ".txt": extrair_txt}


def ingerir(caminho, **kw):
    """Despacha pelo sufixo do arquivo. E este o unico ponto que precisa saber de formatos."""
    ext = os.path.splitext(caminho)[1].lower()
    if ext not in EXTRATORES:
        raise ValueError(f"formato nao suportado: {ext}")
    return EXTRATORES[ext](caminho, **kw) if ext == ".pdf" else EXTRATORES[ext](caminho)


print("Formatos aceitos:", ", ".join(sorted(EXTRATORES)), "+ texto colado")

Formatos aceitos: .docx, .htm, .html, .pdf, .txt + texto colado


### 2.1 Portão de qualidade da extração (passo nosso)

Logo na primeira rodada vimos que nem todo PDF oficial tem camada de texto utilizável, e
que os extratores erram de formas diferentes no mesmo arquivo. Encontramos quatro defeitos:

- glifos sobrepostos, que saem como letras dobradas: *"Plano de Conntingência"*;
- colunas embaralhadas, que picam as palavras em pedaços de uma ou duas letras;
- caixas de texto intercaladas, em que os pedaços se misturam sem nenhuma palavra quebrar;
- palavra cortada ao meio: *"Os re veícu conhe como conta"*. Esse foi o mais difícil, porque
  cada pedaço tem tamanho de palavra normal e começa em minúscula, então as outras medidas
  não pegam.

Na cartilha do Pronaf apareceu um quinto caso, que as medidas acima já cobrem: a página da
ficha catalográfica vem com a fonte quebrada e extrai como
`&ŝĐŚĂĐĂƚĂůŽŐƌĄĨŝĐĂĞůĂďŽƌĂĚĂ`.

Se isso entra no corpus, o modelo aprende a escrever errado e a gente só descobre no fim.
Por isso colocamos um portão antes da limpeza, com quatro medidas:

| Medida | O que detecta | Limite |
|---|---|---|
| letras dobradas implausíveis | glifos sobrepostos e fonte quebrada | 6% |
| palavras de 1 ou 2 letras fora da lista de curtas válidas | leitura errada de colunas | 15% |
| maiúscula no meio da frase | caixas de texto intercaladas | 20% |
| palavra que é só o começo de outra palavra do corpus | palavra cortada ao meio | 12% |

A última medida precisa saber quais palavras existem de verdade, então a extração roda em
duas passadas: primeiro juntamos todo o texto e montamos um vocabulário com as palavras que
aparecem pelo menos três vezes, depois avaliamos bloco a bloco. Um pedaço como *"veícu"*
nunca chega a três ocorrências, mas é o começo de *"veículos"*, que chega.

Em cada PDF rodamos os dois extratores e ficamos com o que entrega mais texto aprovado. O
que é barrado vai para uma quarentena que aparece no relatório, com o motivo.

In [5]:
# @title Portao de qualidade da extracao (passo nosso, nao pedido no enunciado)
# Motivo: nem todo PDF oficial tem uma camada de texto utilizavel. Se um bloco ruim
# entra, o estrago so aparece la na frente, ja dentro do corpus, onde e caro achar.
# Entao medimos a saida do extrator ANTES de deixar o texto seguir.

# Em portugues as unicas letras dobradas comuns sao rr, ss e cc (carro, passo, accao).
# Fora ee e oo (veem, voo), duplicacao em volume e sinal de glifos sobrepostos.
DOBRA_IMPLAUSIVEL = re.compile(r"(?i)([abdefghijklmnopqtuvwxyz])\1")
DOBRA_OK = {"ee", "oo"}

# Palavras de 1 a 2 letras legitimas. O que sobrar costuma ser palavra picada por
# leitura errada de layout em duas colunas.
CURTAS_OK = set("""a o e as os da de do na no em um uma ao aos das dos se me te lhe
nos vos eu tu ele ela nao sim ja ha so la ca ai ate com por sem sob sao tem ter ver
ir vai foi era ou que pra pro num numa dai art inc par ii iii iv vi vii viii ix xi""".split())


def metricas_extracao(texto):
    palavras = re.findall(r"[A-Za-zÀ-ÿ]+", texto)
    if not palavras:
        return {"palavras": 0, "taxa_dobra": 0.0, "taxa_fragmento": 0.0,
                "taxa_maiuscula_no_meio": 0.0}
    dobradas = 0
    for p in palavras:
        if any(m.group(0) not in DOBRA_OK for m in DOBRA_IMPLAUSIVEL.finditer(p.lower())):
            dobradas += 1
    fragmentos = sum(1 for p in palavras if len(p) <= 2 and p.lower() not in CURTAS_OK)
    return {"palavras": len(palavras),
            "taxa_dobra": dobradas / len(palavras),
            "taxa_fragmento": fragmentos / len(palavras),
            "taxa_maiuscula_no_meio": taxa_maiuscula_no_meio(texto)}


def taxa_maiuscula_no_meio(texto):
    """Terceiro sintoma, o mais dificil: cartilha diagramada em caixas de texto sai
    com os pedacos intercalados ("Exercicio em que receber pessoas que precisam..."),
    e nenhuma palavra fica quebrada, entao as duas medidas acima nao veem nada.
    O rastro que sobra e a maiuscula no meio da frase: cada pedaco colado traz o
    inicio de outra frase. Em prosa normal isso e so nome proprio, e e raro."""
    tokens = texto.split()
    if len(tokens) < 25:
        return 0.0
    no_meio, fim_de_frase = 0, True
    for t in tokens:
        limpo = re.sub(r"^[^\wÀ-ÿ]+", "", t)
        if re.match(r"[A-ZÀ-Þ][a-zà-ÿ]{2,}", limpo) and not fim_de_frase:
            no_meio += 1
        fim_de_frase = bool(re.search(r"[.!?:;]$", t))
    return no_meio / len(tokens)


# Limiares calibrados nos proprios documentos desta GS: os blocos saudaveis ficaram
# em torno de 1% nas quatro taxas e os corrompidos passaram de 17%. A folga evita
# reprovar bloco bom.
# --------------------------------------------------------------------------
# Quarto sintoma, e o mais teimoso: a leitura em colunas nao so embaralha, ela
# CORTA as palavras no meio ("Os re veicu conhe como conta ruptu"). Nenhuma das
# tres medidas acima ve isso, porque os pedacos tem tamanho de palavra normal e
# comecam em minuscula. O que entrega o defeito e que cada pedaco e o COMECO de
# uma palavra que existe no corpus, sem ser ele mesmo uma palavra do corpus.
# Para decidir isso precisamos do vocabulario, entao a extracao roda em duas
# passadas: primeiro juntamos todo o texto, depois avaliamos bloco a bloco.

def construir_vocabulario(textos, minimo=3):
    """Palavra 'confirmada' e a que aparece pelo menos 3 vezes em todo o material.
    Pedaco de palavra quebrada quase nunca chega la."""
    contagem = Counter()
    for t in textos:
        contagem.update(w.lower() for w in re.findall(r"[A-Za-zÀ-ÿ]{3,}", t))
    confirmadas = {w for w, n in contagem.items() if n >= minimo}
    prefixos = {w[:i] for w in confirmadas for i in range(3, len(w))}
    return confirmadas, prefixos


def taxa_palavra_truncada(texto, vocabulario):
    confirmadas, prefixos = vocabulario
    palavras = [w.lower() for w in re.findall(r"[A-Za-zÀ-ÿ]{3,}", texto)]
    if len(palavras) < 25:
        return 0.0
    truncadas = sum(1 for w in palavras if w not in confirmadas and w in prefixos)
    return truncadas / len(palavras)


LIMITE_DOBRA, LIMITE_FRAGMENTO, LIMITE_MAIUSCULA, LIMITE_TRUNCADA = 0.06, 0.15, 0.20, 0.12
MIN_PARA_AVALIAR = 25  # abaixo disso a taxa oscila demais e o teste perde sentido


def avaliar_bloco(bloco, vocabulario=None):
    """Aprova ou reprova UM bloco. Reprovar o documento inteiro seria caro demais:
    nos PDFs oficiais o estrago costuma estar em poucas paginas diagramadas.
    Blocos curtos (titulos, incisos de lei, itens de lista) passam direto: eles nao
    estao corrompidos, so sao curtos, e a segmentacao vai junta-los com os vizinhos."""
    m = metricas_extracao(bloco["texto"])
    falhas = []
    if m["palavras"] < MIN_PARA_AVALIAR:
        m["ok"], m["motivo"] = True, ""
        return m
    if m["taxa_dobra"] > LIMITE_DOBRA:
        falhas.append(f"letras dobradas {m['taxa_dobra']:.0%}")
    if m["taxa_fragmento"] > LIMITE_FRAGMENTO:
        falhas.append(f"palavras picadas {m['taxa_fragmento']:.0%}")
    if m["taxa_maiuscula_no_meio"] > LIMITE_MAIUSCULA:
        falhas.append(f"texto intercalado {m['taxa_maiuscula_no_meio']:.0%}")
    if vocabulario is not None:
        m["taxa_truncada"] = taxa_palavra_truncada(bloco["texto"], vocabulario)
        if m["taxa_truncada"] > LIMITE_TRUNCADA:
            falhas.append(f"palavras cortadas ao meio {m['taxa_truncada']:.0%}")
    m["ok"] = not falhas
    m["motivo"] = " + ".join(falhas)
    return m


def passar_portao(blocos, vocabulario=None):
    """Devolve (aprovados, quarentena). A quarentena nao e descartada em silencio:
    ela vira linha do relatorio de qualidade."""
    aprovados, quarentena = [], []
    for b in blocos:
        m = avaliar_bloco(b, vocabulario)
        (aprovados if m["ok"] else quarentena).append(
            b if m["ok"] else {**b, "motivo": m["motivo"], "palavras": m["palavras"]})
    return aprovados, quarentena


print("Portao configurado:")
print(f"  letras dobradas  <= {LIMITE_DOBRA:.0%}")
print(f"  palavras picadas <= {LIMITE_FRAGMENTO:.0%}")
print(f"  texto intercalado <= {LIMITE_MAIUSCULA:.0%} (maiuscula no meio da frase)")
print(f"  palavras cortadas ao meio <= {LIMITE_TRUNCADA:.0%} (exige vocabulario, 2a passada)")
print(f"  so avalia blocos com {MIN_PARA_AVALIAR}+ palavras (os curtos passam e sao juntados depois)")

Portao configurado:
  letras dobradas  <= 6%
  palavras picadas <= 15%
  texto intercalado <= 20% (maiuscula no meio da frase)
  palavras cortadas ao meio <= 12% (exige vocabulario, 2a passada)
  so avalia blocos com 25+ palavras (os curtos passam e sao juntados depois)


In [6]:
# @title Etapa 2 - Rodar a extracao em duas passadas
# 1a passada: extrair tudo, com os dois motores nos PDFs, e juntar o vocabulario.
# 2a passada: passar o portao (que agora sabe quais palavras existem de verdade)
# e ficar, em cada PDF, com o motor que entregou mais texto APROVADO. Nao da para
# decidir isso no olho: depende de como cada PDF foi gerado.

# --- 1a passada ------------------------------------------------------------
candidatos = {}
for f in FONTES:
    caminho = os.path.join(PASTA_FONTES, f["arquivo"])
    ext = os.path.splitext(caminho)[1].lower()
    for motor in (["pypdf", "pdfplumber"] if ext == ".pdf" else [None]):
        blocos = extrair_pdf(caminho, motor=motor) if motor else ingerir(caminho)
        candidatos[(f["arquivo"], motor or ext.lstrip("."))] = blocos

# Texto colado: o quinto canal de entrada exigido pelo enunciado.
# Reproduzimos aqui o padrao dos recados que o sindicato rural e a Emater mandam
# no grupo de WhatsApp quando abre o prazo de adesao ao Garantia-Safra. E o texto
# que a nossa persona recebe no celular e cola para perguntar o que significa.
TEXTO_COLADO = """ATENCAO AGRICULTORES FAMILIARES: abriu o prazo de adesao ao Garantia-Safra
desta safra. Quem perdeu lavoura por seca ou por excesso de chuva precisa estar inscrito
ANTES da perda, nao depois.

Para aderir e preciso ter o CAF ativo, estar enquadrado no Pronaf e pagar a cota individual
do produtor no prazo. A adesao e feita pelo municipio, que precisa ter aderido ao programa
e estar com a cota em dia. Culturas cobertas: arroz, algodao, feijao, mandioca e milho.

Leve ao sindicato: documento com foto, CPF, numero do CAF ativo e o comprovante da area
plantada. Quem nao tiver CAF precisa procurar a entidade emissora antes de tudo.

O beneficio so e pago quando a perda verificada passa de 50 por cento da producao das
culturas cobertas, e cada beneficiario recebe no maximo um beneficio por ano-safra.
Duvidas sobre a sua inscricao: procure o sindicato, a Emater ou a secretaria de agricultura
do municipio. Nao existe taxa para se inscrever alem da cota oficial do produtor."""

blocos_colados = extrair_colado(TEXTO_COLADO, rotulo="recado_garantia_safra.txt")
candidatos[("recado_garantia_safra.txt", "colado")] = blocos_colados

VOCABULARIO = construir_vocabulario(b["texto"] for bs in candidatos.values() for b in bs)
print(f"Vocabulario de referencia: {len(VOCABULARIO[0])} palavras confirmadas "
      f"(3 ou mais ocorrencias) e {len(VOCABULARIO[1])} prefixos.")

# --- 2a passada ------------------------------------------------------------
documentos = {}    # arquivo -> blocos aprovados
quarentena = []    # blocos reprovados, com o motivo
relato_ingestao = []

for f in FONTES + [{"arquivo": "recado_garantia_safra.txt", "formato": "colado"}]:
    melhor = None
    for (arquivo, motor), blocos in candidatos.items():
        if arquivo != f["arquivo"]:
            continue
        aprovados, quar = passar_portao(blocos, VOCABULARIO)
        palavras = sum(len(b["texto"].split()) for b in aprovados)
        if melhor is None or palavras > melhor["palavras"]:
            melhor = {"motor": motor, "blocos": blocos, "aprovados": aprovados,
                      "quarentena": quar, "palavras": palavras}

    documentos[f["arquivo"]] = melhor["aprovados"]
    quarentena.extend(melhor["quarentena"])
    relato_ingestao.append({
        "arquivo": f["arquivo"], "formato": f["formato"], "motor": melhor["motor"],
        "blocos": len(melhor["blocos"]), "aprovados": len(melhor["aprovados"]),
        "quarentena": len(melhor["quarentena"]), "palavras": melhor["palavras"]})

titulo("Resultado da ingestao")
tabela([(r["arquivo"][:36], r["formato"], r["motor"], r["blocos"], r["aprovados"],
         r["quarentena"], r["palavras"]) for r in relato_ingestao],
       ["Arquivo", "Fmt", "Motor", "Blocos", "Aprov", "Quar", "Palavras"],
       [36, 6, 11, 6, 6, 5, 9])
print(f"\nTotal: {sum(r['palavras'] for r in relato_ingestao)} palavras aprovadas em "
      f"{len(documentos)} entradas; {len(quarentena)} blocos em quarentena.")

titulo("Quarentena: por que cada bloco foi barrado", "-")
tabela(sorted(Counter(re.sub(r"\d+%", "", q["motivo"]).strip() for q in quarentena).items(),
              key=lambda x: -x[1]), ["Motivo", "Blocos"], [44, 8])
for q in quarentena[:3]:
    print(f"\n{q['fonte']} ({q['local']}) -> {q['motivo']}")
    print("   ", amostra(q["texto"], 190))

Vocabulario de referencia: 1655 palavras confirmadas (3 ou mais ocorrencias) e 4620 prefixos.

Resultado da ingestao
Arquivo                               Fmt     Motor        Blocos  Aprov   Quar   Palavras 
------------------------------------  ------  -----------  ------  ------  -----  ---------
cartilha_pronaf_credito_rural.pdf     PDF     pypdf        44      38      6      9984     
manual_garantia_safra_MAPA.pdf        PDF     pypdf        12      11      1      3299     
modelo_contrato_ater_MDA.docx         DOCX    docx         115     113     2      2123     
lei_10420_2002_garantia_safra.html    HTML    html         132     124     8      4672     
avisos_oficiais_inmet.txt             TXT     txt          7       7       0      788      
recado_garantia_safra.txt             colado  colado       4       4       0      165      

Total: 21031 palavras aprovadas em 6 entradas; 17 blocos em quarentena.

-------------------------------------------------------------------------

## 3. Limpeza e normalização

A limpeza tem contador por regra. Sem isso ela vira caixa-preta e não dá para justificar
nenhuma decisão depois, que é justamente o que o enunciado cobra.

**O que removemos:** cabeçalho e rodapé (detectados por frequência, não por lista escrita à
mão), numeração de página, pontilhado de sumário, URLs soltas, linhas repetidas e as
anotações de vigência da lei (`(Incluído pela Lei nº ...)`), que são legítimas para quem
estuda a norma mas viram ruído para quem só quer saber se tem direito.

**O que corrigimos:** palavra quebrada por hífen no fim da linha e ligaduras tipográficas.

A cartilha do Pronaf trouxe um problema novo: ela foi exportada do InDesign e cada página
carrega a marca do arquivo e a data da exportação, sempre duas vezes coladas
(`ARQUIVO-EDITAVEL-PRONAF.indd 9ARQUIVO-EDITAVEL-PRONAF.indd 9`). O mesmo acontece com os
títulos. Tratamos como três regras separadas: colapsar a duplicação, remover a marca e
remover a data.

O caso mais trabalhoso foi o hífen. A regra óbvia (`palavra-\nresto`) pega só parte: o
`pypdf` junta as linhas com espaço e sobra *"imediata - mente"*, *"comunida - de"*. Juntar
todo `X - y` apagaria travessão legítimo. A saída foi usar o próprio corpus como juiz:
só juntamos quando nenhum dos dois pedaços aparece sozinho no texto e a palavra inteira
aparece.

**O que não fizemos:** não removemos stop words e não aplicamos stemming,
como o enunciado alerta. Essas técnicas servem para busca e classificação. Um modelo de
linguagem precisa do texto inteiro para aprender a escrever. Também mantivemos acentos,
pontuação e maiúsculas: a célula abaixo conta quantos sobreviveram, como prova.

In [7]:
# @title Etapa 3 - Limpeza, com contador por regra
# Cada regra incrementa um contador. Sem isso a limpeza vira caixa-preta e nao da
# para defender nenhuma decisao depois.

contador_limpeza = Counter()

RE_URL = re.compile(r"(?i)\b(?:https?://|www\.)\S+")
RE_EMAIL_SOLTO = re.compile(r"(?i)\b[\w.+-]+@[\w-]+\.[\w.]+\b")
RE_SO_NUMERO = re.compile(r"^\s*[-–—]?\s*\d{1,4}\s*[-–—]?\s*$")
RE_PAGINA = re.compile(r"(?i)^\s*(p[aá]g(?:ina)?\.?\s*)?\d{1,4}\s*(de\s*\d{1,4})?\s*$")
RE_CONTROLE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f�]")
RE_PONTILHADO = re.compile(r"[.·•]{4,}")          # sumarios: "Capitulo 1 ....... 12"
# Marca de diagramacao que o InDesign deixa no PDF ("ARQUIVO-EDITAVEL-PRONAF.indd 9")
# junto com a data e a hora da exportacao. Nao e conteudo, e assinatura de software.
RE_MARCA_INDD = re.compile(r"(?i)\S*\.indd\s*\d*")
RE_DATA_HORA = re.compile(r"\b\d{2}/\d{2}/\d{4}\s+\d{2}:\d{2}:\d{2}")
# O mesmo PDF repete cada marca duas vezes coladas, e faz o mesmo com os titulos
# ("CREDITO RURALCREDITO RURAL"). So colapsamos quando o trecho repetido tem 12 ou
# mais caracteres e alguma maiuscula, que e o padrao dessa duplicacao de diagramacao.
RE_REPETIDO_COLADO = re.compile(r"((?=[^\n]*[A-ZÀ-Þ])[^\n]{12,60}?)\1")
# Anotacao de vigencia da lei: "(Incluido pela Lei no 14.750, de 2023)". E informacao
# legitima para quem estuda a lei, mas para um assistente de cidadao e ruido que ainda
# por cima carrega numeros que confundem o verificador de fidelidade.
RE_ANOTACAO_LEI = re.compile(
    r"\((?:Inclu[ií]d[oa]|Reda[çc][ãa]o dada|Revogad[oa]|Vide|Regulamento|Vig[êe]ncia)[^)]{0,80}\)")
# O hifen de fim de linha as vezes vem com espaco antes ("imediata - mente"), por
# isso o \s? opcional. O \n continua obrigatorio: sem ele juntariamos palavras
# legitimamente separadas por travessao no meio da frase.
RE_HIFEN_QUEBRA = re.compile(r"(\w{2,})\s?[-‐‑]\s*\n\s*([a-zà-ÿ]\w*)")

# Ligaduras tipograficas: um glifo unico para "fi", "fl" etc. Se nao desfizer, o
# tokenizer ve uma palavra que nao existe. No PDF do MMA o pypdf devolve 37 ligaduras
# e o pdfplumber nenhuma; como o motor vencedor pode mudar de documento para
# documento, a regra fica ligada mesmo quando o contador sai zerado.
LIGADURAS = {"ﬁ": "fi", "ﬂ": "fl", "ﬀ": "ff", "ﬃ": "ffi", "ﬄ": "ffl", "ﬅ": "ft", "ﬆ": "st"}


def detectar_cabecalhos_rodapes(blocos, limiar=0.25, max_palavras=12):
    """Cabecalho e rodape sao a mesma linha curta repetida em muitas paginas.
    Nao da para listar a mao documento por documento, entao deduzimos pela frequencia."""
    freq = Counter()
    for b in blocos:
        for linha in {l.strip() for l in b["texto"].split("\n") if l.strip()}:
            if len(linha.split()) <= max_palavras:
                freq[linha] += 1
    minimo = max(2, int(limiar * len(blocos)))
    return {linha for linha, n in freq.items() if n >= minimo}


def limpar_texto(texto, repetidas=frozenset()):
    for lig, sub in LIGADURAS.items():
        if lig in texto:
            contador_limpeza["ligaduras"] += texto.count(lig)
            texto = texto.replace(lig, sub)

    # Juntar palavra quebrada por hifen no fim da linha (etapa 3 do enunciado).
    texto, n = RE_HIFEN_QUEBRA.subn(r"\1\2", texto)
    contador_limpeza["hifen_quebrado"] += n

    texto, n = RE_CONTROLE.subn("", texto)
    contador_limpeza["caracteres_estranhos"] += n
    texto, n = RE_PONTILHADO.subn(" ", texto)
    contador_limpeza["sumario_pontilhado"] += n
    texto, n = RE_REPETIDO_COLADO.subn(r"\1", texto)
    contador_limpeza["duplicacao_de_diagramacao"] += n
    texto, n = RE_MARCA_INDD.subn(" ", texto)
    contador_limpeza["marca_de_diagramacao"] += n
    texto, n = RE_DATA_HORA.subn(" ", texto)
    contador_limpeza["data_hora_de_exportacao"] += n
    texto, n = RE_ANOTACAO_LEI.subn("", texto)
    contador_limpeza["anotacao_de_vigencia"] += n
    texto, n = RE_URL.subn("", texto)
    contador_limpeza["urls_soltas"] += n

    saida, anterior = [], None
    for linha in texto.split("\n"):
        l = linha.strip()
        if not l:
            saida.append("")
            continue
        if l in repetidas:
            contador_limpeza["cabecalho_rodape"] += 1
            continue
        if RE_SO_NUMERO.match(l) or RE_PAGINA.match(l):
            contador_limpeza["numeracao_pagina"] += 1
            continue
        if l == anterior:
            contador_limpeza["linhas_repetidas"] += 1
            continue
        saida.append(l)
        anterior = l
    return "\n".join(saida)


documentos_limpos = {}
for arquivo, blocos in documentos.items():
    repetidas = detectar_cabecalhos_rodapes(blocos)
    novos = []
    for b in blocos:
        t = limpar_texto(b["texto"], repetidas)
        if t.strip():
            novos.append({**b, "texto": t})
    documentos_limpos[arquivo] = novos

# Segunda passada: hifen de fim de linha que ja chegou sem a quebra de linha.
# O pypdf junta as linhas com espaco, entao "te-\nnham" vira "te - nham" e a regra
# anterior nao pega. Juntar todo "X - y" seria perigoso (quebraria travessao de
# verdade), entao usamos o proprio corpus como juiz: so juntamos quando nenhum dos
# dois pedacos existe sozinho no texto e a palavra inteira existe.
_vocab_corpus = Counter()
for _bs in documentos_limpos.values():
    for _b in _bs:
        _vocab_corpus.update(w.lower() for w in re.findall(r"[A-Za-zÀ-ÿ]{2,}", _b["texto"]))

RE_HIFEN_SOLTO = re.compile(r"\b([A-Za-zÀ-ÿ]{2,})\s+-\s+([a-zà-ÿ]{2,})\b")


def juntar_hifens_soltos(texto):
    def _sub(m):
        esq, dir_ = m.group(1), m.group(2)
        inteira = (esq + dir_).lower()
        pedaco_sozinho = _vocab_corpus.get(esq.lower(), 0) >= 3 and _vocab_corpus.get(dir_, 0) >= 3
        if not pedaco_sozinho and _vocab_corpus.get(inteira, 0) >= 1:
            contador_limpeza["hifen_sem_quebra"] += 1
            return esq + dir_
        return m.group(0)
    return RE_HIFEN_SOLTO.sub(_sub, texto)


documentos_limpos = {a: [{**b, "texto": juntar_hifens_soltos(b["texto"])} for b in bs]
                     for a, bs in documentos_limpos.items()}

titulo("O que a limpeza removeu ou corrigiu")
tabela(sorted(contador_limpeza.items(), key=lambda x: -x[1]), ["Regra", "Ocorrencias"], [26, 12])

_antes = sum(len(b["texto"]) for bs in documentos.values() for b in bs)
_depois = sum(len(b["texto"]) for bs in documentos_limpos.values() for b in bs)
print(f"\nCaracteres: {_antes} -> {_depois} ({(1 - _depois / _antes):.1%} removidos)")


O que a limpeza removeu ou corrigiu
Regra                       Ocorrencias 
--------------------------  ------------
hifen_quebrado              94          
cabecalho_rodape            78          
anotacao_de_vigencia        70          
duplicacao_de_diagramacao   51          
marca_de_diagramacao        45          
data_hora_de_exportacao     45          
urls_soltas                 5           
caracteres_estranhos        0           
sumario_pontilhado          0           

Caracteres: 140075 -> 125975 (10.1% removidos)


In [8]:
# @title Etapa 4 - Normalizacao (sem mexer em acento, pontuacao nem maiuscula)
# Aqui e facil exagerar. Baixar tudo para minusculo ou tirar acento deixaria o
# corpus "limpo" e o modelo burro: ele precisa aprender a escrever "voce nao deve
# atravessar" com acento e com a primeira letra maiuscula.

ASPAS = {"“": '"', "”": '"', "„": '"', "«": '"', "»": '"',
         "‘": "'", "’": "'", "‚": "'", "′": "'", "″": '"'}
TRACOS = {"‐": "-", "‑": "-", "‒": "-", "–": "-", "—": "-", "−": "-"}
ESPACOS = {" ": " ", " ": " ", " ": " ", " ": " ", "​": "", "﻿": ""}

contador_norm = Counter()


def normalizar(texto):
    # NFC junta "a" + acento num unico caractere. Sem isso, "ação" pode virar duas
    # sequencias diferentes de bytes que o tokenizer trata como palavras distintas.
    texto = unicodedata.normalize("NFC", texto)
    for tabela_sub, rotulo in ((ASPAS, "aspas"), (TRACOS, "tracos"), (ESPACOS, "espacos")):
        for de, para in tabela_sub.items():
            if de in texto:
                contador_norm[rotulo] += texto.count(de)
                texto = texto.replace(de, para)
    texto = re.sub(r"[ \t]+", " ", texto)          # espacos repetidos
    texto = re.sub(r" *\n *", "\n", texto)         # espaco colado na quebra
    texto = re.sub(r"\n{3,}", "\n\n", texto)       # no maximo uma linha em branco
    return texto.strip()


documentos_norm = {arq: [{**b, "texto": normalizar(b["texto"])} for b in blocos]
                   for arq, blocos in documentos_limpos.items()}
documentos_norm = {a: [b for b in bs if b["texto"]] for a, bs in documentos_norm.items()}

titulo("Normalizacao aplicada")
tabela(sorted(contador_norm.items(), key=lambda x: -x[1]) or [("nada a padronizar", 0)],
       ["Item padronizado", "Ocorrencias"], [26, 12])

# Prova de que acento e maiuscula continuam la (o enunciado exige isso).
_todo = " ".join(b["texto"] for bs in documentos_norm.values() for b in bs)
_acentos = sum(_todo.count(c) for c in "áàâãéêíóôõúçÁÀÂÃÉÊÍÓÔÕÚÇ")
_maiusculas = sum(1 for c in _todo if c.isupper())
print(f"\nVerificacao: {_acentos} caracteres acentuados e {_maiusculas} maiusculas preservados.")
print(f"Codificacao de saida: UTF-8 (forma NFC).")


Normalizacao aplicada
Item padronizado            Ocorrencias 
--------------------------  ------------
tracos                      36          
aspas                       35          
espacos                     13          

Verificacao: 4077 caracteres acentuados e 9254 maiusculas preservados.
Codificacao de saida: UTF-8 (forma NFC).


In [9]:
# @title Antes e depois da limpeza, no mesmo trecho
# Escolhemos automaticamente o bloco onde a limpeza mais mudou o texto, para
# mostrar o pior caso em vez de um exemplo escolhido a dedo.

_pares = []
for arq in documentos:
    for antes, depois in zip(documentos[arq], documentos_norm.get(arq, [])):
        if antes["local"] == depois["local"] and len(antes["texto"]) > 400:
            delta = len(antes["texto"]) - len(depois["texto"])
            _pares.append((delta, arq, antes["local"], antes["texto"], depois["texto"]))

for _delta, _arq, _local, _a, _d in sorted(_pares, key=lambda x: -x[0])[:2]:
    titulo(f"{_arq} ({_local}) - {_delta} caracteres a menos")
    print("ANTES:")
    print(textwrap.indent(amostra(_a, 430), "  "))
    print("\nDEPOIS:")
    print(textwrap.indent(amostra(_d, 430), "  "))


manual_garantia_safra_MAPA.pdf (p.7) - 871 caracteres a menos
ANTES:
  b) Apresentar, referente ao 1° titular da DAP, comprovante de residência no município cadastrado na
  DAP durante o período da safra em questão (contas de consumo de água, energia elétrica, telefone
  fixo ou móvel e/ou declaração do sindicato de que mora no município, declaração de endereço e
  domícilio feita em cartório no período citado). EXEMPLO: Safra 2015/2016: comprovar que residiu no
  município declarado na DAP no período en

DEPOIS:
  município cadastrado na DAP durante o período da safra em questão (contas de sindicato de que mora
  no município, declaração de endereço e domícilio feita em EXEMPLO: Safra 2015/2016: comprovar que
  residiu no município declarado na Safra 2017/2018: comprovar que residiu no município declarado na
  5. COMO RESOLVER CASOS DE SITUAÇÕES DE RISCO RELACIONADAS AO PERFIL DOS BENEFICIÁRIOS, QUANDO O(A)
  INDÍCIO: DETENTORES DE EMPREGO/CARG

manual_garantia_safra_MAPA.pdf (p.6) -

## 4. Anonimização e LGPD

Foi a etapa em que mais mudamos de ideia no meio do caminho.

### O risco específico deste tema

Um anonimizador comum mascara qualquer coisa parecida com telefone ou e-mail. Só que aqui o
que o cidadão mais precisa achar é justamente um telefone e um e-mail: o 0800 do
Disque-Intoxicação, o 135 do INSS, a caixa de contato da Anater por onde se descobre a
Emater do estado. Mascarar isso deixa o modelo com a forma da resposta e sem o conteúdo.

Separamos em três níveis, e a célula de teste mostra os três lado a lado:

| Nível | Exemplo | O que fazemos |
|---|---|---|
| claramente público | `0800 722 6001`, `135`, `158` | preserva |
| claramente pessoal | celular de plantão de um servidor | mascara |
| ambíguo | fixo institucional, `(61) 3521-5801` | mascara, e explicamos por quê |

O ambíguo mascaramos de propósito: pelo formato não dá para separar de um telefone
residencial, e na dúvida preferimos seguir a LGPD. Com o e-mail dá para fazer melhor, porque
a caixa de serviço se identifica no nome: `contato@`, `ouvidoria@` e `atendimento@` são de
órgão, `joao.silva@` é de pessoa.

### O que detectamos, e como

CPF e CNPJ com conferência do dígito verificador (para não mascarar número de processo),
e-mail, telefone, CEP e nome de pessoa.

Nome foi o mais difícil, e o erro apareceu no corpus desta GS: o `pt_core_news_sm` marcou
"PRONAF MULHER" como pessoa e trocou a linha de crédito inteira por `[NOME]`. Em vez de
escrever uma lista de exceções, usamos dois critérios tirados do próprio corpus:

1. Sobrenome de pessoa quase nunca aparece em minúscula. "crédito" e "familiar" aparecem;
   "Japiassú" não aparece nunca.
2. Nome de pessoa não se repete. Numa cartilha, o nome de alguém sai uma ou duas vezes;
   "Pronaf Mulher" e "Garantia-Safra" saem o tempo todo. Acima de três ocorrências, tratamos
   como nome de programa.

### Como sabemos que funciona

Rodar sem erro não prova nada, porque um anonimizador que não acha nada gera o mesmo
relatório de um que funciona. Então plantamos dados pessoais falsos em frases reais do
corpus e medimos quantos ele recupera, usando de propósito formas de escrita que as nossas
regex não previam. As duas rodadas aparecem na célula, lado a lado.

In [10]:
# @title Carregar o reconhecedor de entidades em portugues
# Regex pega CPF, e-mail, telefone e CEP. Nome de pessoa nao tem formato fixo,
# entao aqui precisamos de modelo.


def carregar_ner():
    try:
        import spacy
        return spacy.load("pt_core_news_sm")
    except Exception as e:
        print(f"(aviso) spaCy indisponivel ({type(e).__name__}). "
              f"Nomes nao serao detectados; rode a celula de instalacao.")
        return None


NLP = carregar_ner()
print("NER carregado:", "pt_core_news_sm" if NLP else "nenhum")

NER carregado: pt_core_news_sm


In [11]:
# @title Filtro de nomes: o NER sozinho nao serve
# O pt_core_news_sm marca como PESSOA coisas como "Planos de Contingencia",
# "Cenario de Risco" e "Secao II". Trocar isso por [NOME] apagaria justamente o
# vocabulario do tema. Precisavamos de um criterio que nao fosse uma lista escrita
# a mao (que so funciona nestes documentos).
#
# A ideia: um sobrenome de pessoa quase nunca aparece em minuscula no corpus.
# "contingencia", "risco", "secao" e "bombeiros" aparecem; "Japiassu" e "Trajber"
# nunca. Entao o proprio corpus vira o dicionario que separa nome proprio de
# substantivo comum capitalizado.

VOCAB_MINUSCULO = Counter()
for _bs in documentos_norm.values():
    for _b in _bs:
        VOCAB_MINUSCULO.update(re.findall(r"[a-zà-ÿ]{3,}", _b["texto"]))

# Segundo criterio, pelo mesmo raciocinio: nome de pessoa aparece uma ou duas vezes
# num documento; nome de programa ("Pronaf Mulher", "Garantia-Safra") aparece o
# tempo todo. Sem isso o NER troca a linha de credito inteira por [NOME].
TEXTO_DO_CORPUS = " ".join(b["texto"] for bs in documentos_norm.values() for b in bs)
MAX_REPETICOES_DE_NOME = 3

PARTICULAS = {"de", "da", "do", "dos", "das", "e", "del", "van", "von", "y"}

# Palavra que ABRE um nome de orgao, cargo ou divisao de texto. Se a entidade
# comeca com uma delas, nao e pessoa.
ABREM_INSTITUICAO = {
    "universidade", "secretaria", "superintendencia", "superintendência", "subsecretaria",
    "ministerio", "ministério", "coordenadoria", "instituto", "centro", "secao", "seção",
    "capitulo", "capítulo", "artigo", "anexo", "visao", "visão", "vigencia", "vigência",
    "rota", "gerente", "coordenador", "coordenadora", "diretor", "diretora", "presidente",
    "prefeitura", "governo", "departamento", "divisao", "divisão", "nucleo", "núcleo",
    "comissao", "comissão", "conselho", "agencia", "agência", "fundacao", "fundação",
    "grupo", "equipe", "trabalho", "chat", "plano", "planos", "cenario", "cenário",
    "compromisso", "corpo", "policia", "polícia", "defesa", "protecao", "proteção",
}

LIMIAR_COMUM = 0.5  # metade ou mais das palavras ja vistas em minuscula => nao e nome


def e_nome_de_pessoa(texto_ent):
    partes = [p for p in re.split(r"\s+", texto_ent.strip()) if p]
    conteudo = [p for p in partes if p.lower() not in PARTICULAS]
    if len(conteudo) < 2:                                  # so o primeiro nome: ambiguo demais
        return False
    if any(not p[:1].isupper() for p in conteudo):         # nome proprio vem capitalizado
        return False
    if conteudo[0].lower().strip(".,;:") in ABREM_INSTITUICAO:
        return False
    if TEXTO_DO_CORPUS.count(texto_ent.strip()) >= MAX_REPETICOES_DE_NOME:
        return False
    comuns = sum(1 for p in conteudo
                 if VOCAB_MINUSCULO.get(re.sub(r"[^\wà-ÿ]", "", p.lower()), 0) >= 2)
    return comuns / len(conteudo) < LIMIAR_COMUM


RE_CAIXA_ALTA = re.compile(r"\b[A-ZÀ-Þ]{3,}(?:[ ]+(?:[A-ZÀ-Þ]{3,}|DE|DA|DO|DOS|DAS)){1,4}\b")


def _destacar_caixa_alta(texto):
    """O NER em portugues nao reconhece nome em CAIXA ALTA, e documento oficial e
    cheio deles em assinatura. Como passar para Capitalizado nao muda o tamanho da
    string, podemos rodar o NER nessa copia e usar os mesmos indices no original."""
    return RE_CAIXA_ALTA.sub(lambda m: m.group(0).title(), texto)


LIGACOES_DE_NOME = {"de", "da", "do", "dos", "das", "e"}


def _plausivel_em_caixa_alta(trecho, texto, fim):
    """Regra mais apertada para o que so foi detectado por causa da CAIXA ALTA.
    Assinatura de documento tem 2 ou 3 nomes ligados por "de"/"da"; titulo de verbete
    de glossario ("SOLUCOES BASEADAS NA NATUREZA:") tem outras ligacoes e vem seguido
    de dois-pontos. Sem este aperto, o glossario inteiro vira [NOME]."""
    if texto[fim:fim + 1] == ":":
        return False
    palavras = trecho.split()
    conteudo = [p for p in palavras if p.lower() not in LIGACOES_DE_NOME]
    ligacoes = [p.lower() for p in palavras if p.lower() not in [c.lower() for c in conteudo]]
    return 2 <= len(conteudo) <= 3 and all(l in LIGACOES_DE_NOME for l in ligacoes)


def entidades_pessoa(texto):
    """Devolve (inicio, fim) de cada nome de pessoa. Quebra entidades que o NER
    colou atraves de uma quebra de linha: numa pagina de creditos, duas assinaturas
    seguidas viram uma entidade so."""
    if NLP is None:
        return []
    achados = []
    espelho = _destacar_caixa_alta(texto)
    assert len(espelho) == len(texto)
    faixas_caixa_alta = [(m.start(), m.end()) for m in RE_CAIXA_ALTA.finditer(texto)]

    def veio_da_caixa_alta(ini, fim):
        return any(a <= ini and fim <= b for a, b in faixas_caixa_alta)

    for ent in NLP(espelho).ents:
        if ent.label_ != "PER":
            continue
        desloc = ent.start_char
        for pedaco in ent.text.split("\n"):
            if pedaco.strip() and e_nome_de_pessoa(pedaco):
                ini = desloc + (len(pedaco) - len(pedaco.lstrip()))
                fim = ini + len(pedaco.strip())
                if (not veio_da_caixa_alta(ini, fim)
                        or _plausivel_em_caixa_alta(texto[ini:fim], texto, fim)):
                    achados.append((ini, fim))
            desloc += len(pedaco) + 1
    return achados


print(f"Vocabulario de apoio: {len(VOCAB_MINUSCULO)} palavras minusculas distintas do proprio corpus.")
print("Exemplos de decisao:")
for _t in ["Maria Aparecida da Silva", "Pronaf Mulher", "Garantia-Safra", "Credito Rural",
           "Jose Carlos Ferreira", "Agricultura Familiar", "Secao II"]:
    print(f"  {_t:<28} -> {'NOME (mascara)' if e_nome_de_pessoa(_t) else 'nao e pessoa (mantem)'}")

Vocabulario de apoio: 2784 palavras minusculas distintas do proprio corpus.
Exemplos de decisao:
  Maria Aparecida da Silva     -> NOME (mascara)
  Pronaf Mulher                -> nao e pessoa (mantem)
  Garantia-Safra               -> nao e pessoa (mantem)
  Credito Rural                -> nao e pessoa (mantem)
  Jose Carlos Ferreira         -> NOME (mascara)
  Agricultura Familiar         -> nao e pessoa (mantem)
  Secao II                     -> nao e pessoa (mantem)


In [12]:
# @title Etapa 5 - Anonimizacao (LGPD)
# Duas armadilhas neste tema, e as duas custam caro:
#  1) mascarar de menos deixa dado pessoal entrar no corpus;
#  2) mascarar demais destroi justamente a informacao que o cidadao precisa. Telefone
#     e dado pessoal, mas 135 (INSS) e um 0800 de ouvidoria sao canal de atendimento.
#     Um assistente que aprendeu "ligue para [TELEFONE]" nao serve de nada para quem
#     esta tentando descobrir onde pedir o beneficio.

MASCARAS = {"cpf": "[CPF]", "cnpj": "[CNPJ]", "email": "[EMAIL]",
            "telefone": "[TELEFONE]", "cep": "[CEP]", "nome": "[NOME]"}

# Numero de utilidade publica nao identifica ninguem: fica.
NUMEROS_PUBLICOS = {
    "135",  # INSS, por onde passa a aposentadoria do segurado especial
    "136",  # Disque Saude
    "158",  # Disque Trabalho
    "162",  # Ouvidoria
    "100", "180", "181", "190", "191", "192", "193", "197", "198", "199", "156",
}

# As formas abaixo ja incorporam as falhas que o teste adversarial (celula seguinte)
# encontrou na nossa primeira versao: CPF separado por espaco, e-mail com espaco em
# volta do @ e CEP sem hifen passavam direto.
RE_CPF = re.compile(r"(?<!\d)\d{3}[.\s]?\d{3}[.\s]?\d{3}[-.\s]?\d{2}(?!\d)")
RE_CNPJ = re.compile(r"(?<!\d)\d{2}[.\s]?\d{3}[.\s]?\d{3}/?\d{4}[-.\s]?\d{2}(?!\d)")
RE_EMAIL = re.compile(r"(?i)\b[\w.+-]+\s?@\s?[\w-]+\.[\w.-]+\b")
# Caixa de servico identifica um orgao, nao uma pessoa: contato@anater.org.br e o
# canal pelo qual a agricultora acha a Emater do estado dela. Mascarar isso seria o
# mesmo erro do 0800. Ja joao.silva@mda.gov.br continua sendo dado pessoal.
CAIXAS_DE_SERVICO = ("contato", "atendimento", "ouvidoria", "sac", "faleconosco",
                     "fale.conosco", "imprensa", "suporte", "informacoes", "informacao",
                     "secretaria", "coordenacao", "gabinete", "cgs", "protocolo")


def email_de_servico(endereco):
    local = endereco.split("@")[0].lower()
    return any(s in local for s in CAIXAS_DE_SERVICO)


RE_CEP = re.compile(r"(?<!\d)\d{5}[-\s]\d{3}(?!\d)")
# Oito digitos colados sao ambiguos (podem ser protocolo ou telefone), entao so
# mascaramos quando a palavra CEP aparece antes.
RE_CEP_CONTEXTO = re.compile(r"(?i)\b(CEP:?\s*)(\d{5}\s?\d{3})(?!\d)")
RE_TEL = re.compile(r"(?<!\d)(?:\+55\s?)?(?:\(?\d{2}\)?[\s.-]?)?(?:9\s?)?\d{4}[\s.-]?\d{4}(?!\d)")
RE_TEL_0800 = re.compile(r"(?<![\d-])0800[\s.-]?\d{3}[\s.-]?\d{3,4}(?![\d-])")

MARCA = "\u0001"  # caractere de controle, nao aparece em texto de documento


def _dv_cpf(parcial):
    peso = len(parcial) + 1
    soma = sum(int(d) * (peso - i) for i, d in enumerate(parcial))
    resto = (soma * 10) % 11
    return "0" if resto == 10 else str(resto)


def cpf_valido(txt):
    """Confere o digito verificador. Sem isso, qualquer sequencia de 11 digitos num
    documento tecnico (numero de processo, de protocolo) viraria [CPF]."""
    d = re.sub(r"\D", "", txt)
    if len(d) != 11 or len(set(d)) == 1:
        return False
    return d[9] == _dv_cpf(d[:9]) and d[10] == _dv_cpf(d[:10])


def cnpj_valido(txt):
    d = re.sub(r"\D", "", txt)
    if len(d) != 14 or len(set(d)) == 1:
        return False
    for tam in (12, 13):
        pesos = ([5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2] if tam == 12
                 else [6, 5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2])
        soma = sum(int(x) * p for x, p in zip(d[:tam], pesos))
        dv = 0 if soma % 11 < 2 else 11 - soma % 11
        if int(d[tam]) != dv:
            return False
    return True


def _telefones(texto, c):
    """0800 nunca e telefone de pessoa: e linha de atendimento de orgao ou de banco.
    Na primeira versao desta ferramenta nos mascaravamos 0800 junto com o resto, e
    isso apagava justamente o canal pelo qual o agricultor resolveria o problema.
    Entao guardamos os 0800, rodamos a regra geral e devolvemos cada um no lugar."""
    guardados = []

    def _guardar(m):
        guardados.append(m.group(0))
        c["telefone_publico_preservado"] += 1
        return f"{MARCA}{len(guardados) - 1}{MARCA}"

    texto = RE_TEL_0800.sub(_guardar, texto)

    def _mascarar(m):
        if re.sub(r"\D", "", m.group(0)) in NUMEROS_PUBLICOS:
            c["telefone_publico_preservado"] += 1
            return m.group(0)
        c["telefone"] += 1
        return MASCARAS["telefone"]

    texto = RE_TEL.sub(_mascarar, texto)
    for i, original in enumerate(guardados):
        texto = texto.replace(f"{MARCA}{i}{MARCA}", original)
    return texto


def anonimizar(texto, c=None):
    c = Counter() if c is None else c

    def aplicar(regex, rotulo, validador=None):
        def _sub(m):
            if validador and not validador(m.group(0)):
                return m.group(0)
            c[rotulo] += 1
            return MASCARAS[rotulo]
        return regex.sub(_sub, texto)

    def _email(m):
        if email_de_servico(m.group(0)):
            c["email_de_servico_preservado"] += 1
            return m.group(0)
        c["email"] += 1
        return MASCARAS["email"]

    texto = RE_EMAIL.sub(_email, texto)
    texto = aplicar(RE_CNPJ, "cnpj", cnpj_valido)
    texto = aplicar(RE_CPF, "cpf", cpf_valido)

    def _cep_com_contexto(m):
        c["cep"] += 1
        return m.group(1) + MASCARAS["cep"]

    texto = RE_CEP_CONTEXTO.sub(_cep_com_contexto, texto)
    texto = aplicar(RE_CEP, "cep")
    texto = _telefones(texto, c)

    # Nomes por ultimo e de tras para frente, para nao invalidar os indices.
    for ini, fim in sorted(entidades_pessoa(texto), reverse=True):
        c["nome"] += 1
        texto = texto[:ini] + MASCARAS["nome"] + texto[fim:]
    return texto, c


contador_anon = Counter()
documentos_anon = {arq: [{**b, "texto": anonimizar(b["texto"], contador_anon)[0]} for b in blocos]
                   for arq, blocos in documentos_norm.items()}

titulo("Dados pessoais encontrados e mascarados nos documentos oficiais")
tabela(sorted(contador_anon.items(), key=lambda x: -x[1]) or [("nada encontrado", 0)],
       ["Tipo", "Ocorrencias"], [36, 12])


Dados pessoais encontrados e mascarados nos documentos oficiais
Tipo                                  Ocorrencias 
------------------------------------  ------------
nome                                  26          
email_de_servico_preservado           1           
telefone                              1           


In [13]:
# @title Antes e depois da anonimizacao (exigido pelo enunciado)
# Nao usamos exemplo inventado. Os trechos abaixo sao blocos reais dos documentos
# oficiais: paginas de credito com nomes de servidores e, no modelo do Parana, os
# celulares de plantao. E exatamente o tipo de dado que nao pode entrar num corpus.

_casos = []
for arq, blocos in documentos_norm.items():
    for b in blocos:
        depois, c = anonimizar(b["texto"])
        achados = sum(v for k, v in c.items() if not k.endswith("preservado"))
        if achados:
            _casos.append({"n": achados, "arq": arq, "local": b["local"],
                           "antes": b["texto"], "depois": depois, "cont": dict(c)})


def _janela(antes, depois, largura=360):
    """Mostra a regiao onde a mascara entrou, e nao os primeiros caracteres do bloco:
    senao o 'antes' e o 'depois' saem identicos na tela."""
    pos = min((depois.find(m) for m in MASCARAS.values() if m in depois), default=0)
    ini_d = max(0, pos - largura // 3)
    trecho_d = depois[ini_d:ini_d + largura]
    ancora = re.sub(r"\s+", " ", depois[max(0, ini_d - 40):ini_d]).strip()
    ini_a = antes.find(ancora) if ancora and antes.find(ancora) >= 0 else ini_d
    return antes[ini_a:ini_a + largura + 60], trecho_d


# Um caso de nome e um caso de telefone, para cobrir os dois tipos de mascara.
_escolhidos = []
for _tipo in ("telefone", "nome"):
    for _c in sorted(_casos, key=lambda x: -x["cont"].get(_tipo, 0)):
        if _c["cont"].get(_tipo) and _c not in _escolhidos:
            _escolhidos.append(_c)
            break

for _c in _escolhidos:
    _a, _d = _janela(_c["antes"], _c["depois"])
    titulo(f"{_c['arq']} ({_c['local']}) - mascaras: {_c['cont']}")
    print("ANTES:")
    print(textwrap.indent(amostra(_a, 400), "  "))
    print("\nDEPOIS:")
    print(textwrap.indent(amostra(_d, 400), "  "))

print(f"\nBlocos com algum dado pessoal: {len(_casos)} de "
      f"{sum(len(b) for b in documentos_norm.values())}.")
print("Nenhum dado pessoal real segue para os corpus: tudo o que vai para o JSONL "
      "passa por esta etapa antes.")


cartilha_pronaf_credito_rural.pdf (p.17) - mascaras: {'email_de_servico_preservado': 1, 'telefone': 1}
ANTES:
  Para consultar outros telefones das Emateres, é só entrar em contato com a Anater O telefone é: (61)
  3521-5801 O site é: E o correio eletrônico (e-mail) é: contato@anater.org.br fiquem atentas! •
  Ninguém pode cobrar para emitir o CAF! É proibido por lei. • Também é proibido exigir título de
  eleitora das famílias! • As bases de dados do governo são integradas. Estão previstas na lei
  penalidades par

DEPOIS:
  Para consultar outros telefones das Emateres, é só entrar em contato com a Anater O telefone é:
  [TELEFONE] O site é: E o correio eletrônico (e-mail) é: contato@anater.org.br fiquem atentas! •
  Ninguém pode cobrar para emitir o CAF! É proibido por lei. • Também é proibido exigir título de
  eleitora das famílias! • As bases de dados do governo são integradas. Es

cartilha_pronaf_credito_rural.pdf (p.32) - mascaras: {'nome': 3}
ANTES:
  PRONAF BIOECONOMIA Esta l

In [14]:
# @title Teste adversarial da anonimizacao (passo nosso)
# "Rodou sem erro" nao prova nada: um anonimizador que nao acha nada produz o mesmo
# relatorio de um anonimizador perfeito. Entao plantamos dados pessoais falsos em
# frases reais do corpus e medimos quantos ele recupera.
#
# E de proposito o teste usa VARIACOES de escrita que as nossas regex nao foram
# feitas para pegar (CPF sem pontuacao, telefone com espaco no lugar do hifen,
# nome em caixa alta). Testar so o formato que voce mesmo escreveu nao e teste.

random.seed(SEED)

NOMES_FALSOS = ["Marlene Aparecida de Souza", "Joao Batista Ferreira", "Rita Camargo Alves",
                "Sebastiao Nunes Prado", "Ednaldo Tavares Pinto", "Celia Moraes Bittencourt",
                "Wagner Policarpo Mendes", "Jurema Antunes Caldeira"]


def _cpf_digitos():
    base = "".join(str(random.randint(0, 9)) for _ in range(9))
    while len(set(base)) == 1:
        base = "".join(str(random.randint(0, 9)) for _ in range(9))
    base += _dv_cpf(base)
    return base + _dv_cpf(base)


def variantes(tipo):
    """Cada tipo vira varias formas de escrita, como aparece no mundo real."""
    if tipo == "cpf":
        d = _cpf_digitos()
        return [f"{d[:3]}.{d[3:6]}.{d[6:9]}-{d[9:]}", d, f"{d[:3]} {d[3:6]} {d[6:9]} {d[9:]}"]
    if tipo == "email":
        u = random.choice(["marlene", "joao.batista", "rita_alves", "ednaldo77"])
        dom = random.choice(["gmail.com", "bol.com.br", "defesacivil.sp.gov.br"])
        return [f"{u}@{dom}", f"{u.upper()}@{dom}", f"{u} @ {dom}"]
    if tipo == "telefone":
        ddd, p1, p2 = random.randint(11, 99), random.randint(90000, 99999), random.randint(1000, 9999)
        return [f"({ddd}) {p1}-{p2}", f"{ddd} {p1} {p2}", f"({ddd}){p1}-{p2}", f"+55 {ddd} {p1}{p2}"]
    if tipo == "cep":
        a, b = random.randint(10000, 99999), random.randint(100, 999)
        return [f"{a}-{b}", f"{a}{b}", f"{a} {b}"]
    n = random.choice(NOMES_FALSOS)
    return [n, n.upper(), n.replace(" de ", " De ")]


MOLDES = {
    "cpf": "O morador informou o CPF {v} no cadastro do abrigo.",
    "email": "Duvidas podem ser enviadas para {v} durante o periodo chuvoso.",
    "telefone": "O contato da familia cadastrada e {v}.",
    "cep": "O imovel fica no CEP {v}, em area de risco mapeada.",
    "nome": "A vistoria foi acompanhada por {v}, moradora do setor 3.",
}

# ---------------------------------------------------------------------------
# Para o teste servir de alguma coisa, ele tem que comparar duas versoes.
# A v1 e a que escrevemos antes de rodar o teste: regex "do jeito do livro",
# sem espaco alternativo e sem tratar CAIXA ALTA. A v2 e a que esta nas celulas
# acima, ja corrigida pelo que este teste apontou.
RE_CPF_V1 = re.compile(r"\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b")
RE_EMAIL_V1 = re.compile(r"(?i)\b[\w.+-]+@[\w-]+\.[\w.-]+\b")
RE_CEP_V1 = re.compile(r"\b\d{5}-\d{3}\b")


def anonimizar_v1(texto):
    texto = RE_EMAIL_V1.sub(MASCARAS["email"], texto)
    texto = RE_CPF_V1.sub(lambda m: MASCARAS["cpf"] if cpf_valido(m.group(0)) else m.group(0), texto)
    texto = RE_CEP_V1.sub(MASCARAS["cep"], texto)
    texto = RE_TEL.sub(lambda m: m.group(0) if re.sub(r"\D", "", m.group(0)) in NUMEROS_PUBLICOS
                       else MASCARAS["telefone"], texto)
    if NLP is not None:  # sem o truque da CAIXA ALTA
        for ent in sorted([e for e in NLP(texto).ents if e.label_ == "PER"],
                          key=lambda e: -e.start_char):
            if e_nome_de_pessoa(ent.text):
                texto = texto[:ent.start_char] + MASCARAS["nome"] + texto[ent.end_char:]
    return texto


_frases = [b["texto"].split("\n")[0] for bs in documentos_norm.values()
           for b in bs if 60 < len(b["texto"].split("\n")[0]) < 220]
random.shuffle(_frases)

N_POR_TIPO = 60
acertos, acertos_v1, totais = Counter(), Counter(), Counter()
escapou = defaultdict(list)

for tipo in MOLDES:
    for i in range(N_POR_TIPO):
        valor = random.choice(variantes(tipo))
        texto = f"{_frases[(i * 7) % len(_frases)]} {MOLDES[tipo].format(v=valor)}"
        totais[tipo] += 1
        saida = anonimizar(texto)[0]
        if MASCARAS[tipo] in saida and valor not in saida:
            acertos[tipo] += 1
        elif len(escapou[tipo]) < 3:
            escapou[tipo].append(valor)
        saida1 = anonimizar_v1(texto)
        if MASCARAS[tipo] in saida1 and valor not in saida1:
            acertos_v1[tipo] += 1

titulo("Recall sobre dados pessoais plantados, antes e depois de corrigir o anonimizador")
tabela([(t, totais[t], f"{acertos_v1[t] / totais[t]:.0%}", f"{acertos[t] / totais[t]:.0%}",
         "; ".join(escapou[t]) or "-") for t in MOLDES],
       ["Tipo", "Plantados", "Recall v1", "Recall v2", "Ainda escapa"], [10, 10, 10, 10, 40])
print(f"\nGeral: v1 {sum(acertos_v1.values()) / sum(totais.values()):.1%}  ->  "
      f"v2 {sum(acertos.values()) / sum(totais.values()):.1%}  "
      f"({sum(totais.values())} injecoes)")
print("O que a v1 perdia: CPF separado por espaco, e-mail com espaco em volta do @,")
print("CEP sem hifen e nome escrito em CAIXA ALTA, comum em assinatura de documento oficial.")

# Falso positivo: o anonimizador nao pode estragar texto sem dado pessoal.
_limpas = _frases[:150]
_estragadas = [f for f in _limpas if any(m in anonimizar(f)[0] for m in MASCARAS.values())]
print(f"\nFrases sem PII plantada que mesmo assim foram alteradas: {len(_estragadas)} de "
      f"{len(_limpas)} ({len(_estragadas) / len(_limpas):.1%}).")
for _f in _estragadas[:2]:
    print("   original:", amostra(_f, 120))
    print("   saida   :", amostra(anonimizar(_f)[0], 120))
if _estragadas:
    print("   (vale olhar cada caso antes de chamar de erro: em documento oficial costuma")
    print("    ser assinatura, ou seja, nome real, e o anonimizador acertou.)")

# O ponto que mais importa neste tema: separar canal publico de dado pessoal.
# Estes documentos nao citam 0800 nem 135, entao aqui e um teste controlado: as
# quatro frases abaixo foram escritas por nos justamente para cobrir os tres casos.
titulo("Canal de atendimento, telefone pessoal e o caso ambiguo", "-")
CASOS_DE_TELEFONE = [
    ("inequivocamente publico", "Em caso de intoxicacao ligue 0800 722 6001, o Disque-Intoxicacao."),
    ("inequivocamente publico", "Para falar sobre a aposentadoria rural, ligue 135, do INSS."),
    ("inequivocamente pessoal", "O celular de plantao do tecnico e (61) 99876-5432."),
    ("ambiguo: fixo institucional", "O telefone da Anater e (61) 3521-5801."),
]
RE_INGENUO = re.compile(r"\b\d{3,}\b")  # o atalho que muita gente escreve
for _tipo, _frase in CASOS_DE_TELEFONE:
    print(f"\n  [{_tipo}]")
    print(f"    original      : {_frase}")
    print(f"    nosso         : {anonimizar(_frase)[0]}")
    print(f"    regra ingenua : {RE_INGENUO.sub('[TELEFONE]', _frase)}")
print("\n  O fixo institucional nos mascaramos de proposito: pelo formato nao da para")
print("  distinguir de um telefone residencial, e na duvida a LGPD pesa mais que a")
print("  conveniencia. O 0800 e os numeros curtos de servico nao tem essa ambiguidade.")

titulo("E o e-mail?", "-")
for _e in ["contato@anater.org.br", "ouvidoria@mda.gov.br", "joao.silva@mda.gov.br",
           "mariadasilva@gmail.com"]:
    _saida = anonimizar(f"Escreva para {_e} se tiver duvida.")[0]
    _resultado = _saida.replace("Escreva para ", "").replace(" se tiver duvida.", "")
    print(f"  {_e:<26} -> {_resultado}")
print("  Caixa de servico identifica orgao; caixa com nome de pessoa identifica pessoa.")


Recall sobre dados pessoais plantados, antes e depois de corrigir o anonimizador
Tipo        Plantados   Recall v1   Recall v2   Ainda escapa                            
----------  ----------  ----------  ----------  ----------------------------------------
cpf         60          60%         100%        -                                       
email       60          72%         100%        -                                       
telefone    60          100%        100%        -                                       
cep         60          30%         100%        -                                       
nome        60          95%         100%        -                                       

Geral: v1 71.3%  ->  v2 100.0%  (300 injecoes)
O que a v1 perdia: CPF separado por espaco, e-mail com espaco em volta do @,
CEP sem hifen e nome escrito em CAIXA ALTA, comum em assinatura de documento oficial.



Frases sem PII plantada que mesmo assim foram alteradas: 1 de 123 (0.8%).
   original: FERNANDO HENRIQUE CARDOSO Pedro Malan Guilherme Gomes Dias José Abrão
   saida   : [NOME]
   (vale olhar cada caso antes de chamar de erro: em documento oficial costuma
    ser assinatura, ou seja, nome real, e o anonimizador acertou.)

------------------------------------------------------------------------------
Canal de atendimento, telefone pessoal e o caso ambiguo
------------------------------------------------------------------------------

  [inequivocamente publico]
    original      : Em caso de intoxicacao ligue 0800 722 6001, o Disque-Intoxicacao.
    nosso         : Em caso de intoxicacao ligue 0800 722 6001, o Disque-Intoxicacao.
    regra ingenua : Em caso de intoxicacao ligue [TELEFONE] [TELEFONE] [TELEFONE], o Disque-Intoxicacao.

  [inequivocamente publico]
    original      : Para falar sobre a aposentadoria rural, ligue 135, do INSS.
    nosso         : Para falar sobre a aposent

## 5. Segmentação

O enunciado pede trechos de 256 a 512 tokens. Token não é palavra, e a diferença não é
pequena: neste corpus o tokenizer do Qwen2.5 gasta cerca de 2 tokens por palavra em
português, e "agricultura" sozinho vira vários pedaços. Contar palavra e torcer para dar
certo colocaria boa parte dos trechos fora da faixa pedida.

Como já sabemos para qual modelo este corpus vai, usamos o tokenizer dele para contar.
É a mesma conta que o fine-tuning vai fazer.

A regra de corte respeita o que o enunciado pede: empilhamos frases até passar de 256
tokens e fechamos o trecho ali, no fim da frase. Se uma frase sozinha passar de 512, ela é
dividida na pontuação fraca antes de qualquer corte seco. Blocos curtos (incisos de lei,
itens de lista, cláusulas do contrato) não são descartados: são costurados com os vizinhos,
e é isso que faz o texto voltar a ter sentido corrido.

In [15]:
# @title Etapa 6 - Contar token com o tokenizer do modelo alvo
# O enunciado pede trechos de 256 a 512 TOKENS. Token nao e palavra: em portugues
# o Qwen2.5 quebra "agricultura" em varias pecas e "a" em uma. Contar palavra e
# chutar. Como ja sabemos para qual modelo esse corpus vai, usamos o tokenizer dele.

TOKENIZER = None
try:
    from transformers import AutoTokenizer
    TOKENIZER = AutoTokenizer.from_pretrained(SLM_ALVO)
    print(f"Tokenizer de {SLM_ALVO} carregado (vocabulario: {TOKENIZER.vocab_size}).")
except Exception as e:
    print(f"(aviso) nao foi possivel baixar o tokenizer ({type(e).__name__}).")
    print("        Usando estimativa por palavras; os limites ficam aproximados.")


def contar_tokens(texto):
    if TOKENIZER is not None:
        return len(TOKENIZER(texto, add_special_tokens=False)["input_ids"])
    return int(len(texto.split()) * 1.9)  # fator medido abaixo, para o caso de falha


if TOKENIZER is not None:
    _amostras = [b["texto"] for bs in documentos_anon.values() for b in bs if len(b["texto"].split()) > 40][:200]
    _pal = sum(len(t.split()) for t in _amostras)
    _tok = sum(contar_tokens(t) for t in _amostras)
    titulo("Por que nao da para contar palavra no lugar de token")
    print(f"Em {len(_amostras)} blocos deste corpus: {_pal} palavras viram {_tok} tokens "
          f"({_tok / _pal:.2f} tokens por palavra).")
    print("Ou seja, um trecho de 300 palavras passaria dos 512 tokens pedidos pelo enunciado.")
    for _t in ["agricultura", "financiamento", "Garantia-Safra", "Pronaf", "lavoura", "135"]:
        print(f"  {_t:<16} -> {TOKENIZER.tokenize(_t)}")

C:\Users\claud\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokenizer de Qwen/Qwen2.5-0.5B-Instruct carregado (vocabulario: 151643).

Por que nao da para contar palavra no lugar de token
Em 84 blocos deste corpus: 15784 palavras viram 31975 tokens (2.03 tokens por palavra).
Ou seja, um trecho de 300 palavras passaria dos 512 tokens pedidos pelo enunciado.
  agricultura      -> ['ag', 'ric', 'ult', 'ura']
  financiamento    -> ['fin', 'anc', 'i', 'amento']
  Garantia-Safra   -> ['Gar', 'ant', 'ia', '-S', 'af', 'ra']
  Pronaf           -> ['P', 'ron', 'af']
  lavoura          -> ['l', 'avour', 'a']
  135              -> ['1', '3', '5']


In [16]:
# @title Etapa 6 - Segmentacao em trechos de 256 a 512 tokens
# Regra: nunca cortar no meio da frase. Vamos empilhando frases ate passar de 256
# tokens e fechamos o trecho; se uma frase sozinha estourar 512, ela e dividida por
# pontuacao secundaria (ponto e virgula, dois pontos) antes de qualquer corte seco.

RE_FRASE = re.compile(r"(?<=[.!?:;])\s+(?=[A-ZÀ-Þ0-9])|\n+")
# Abreviacoes que nao terminam frase. Sem isso "Art. 2o" vira dois trechos.
ABREV = re.compile(r"(?i)\b(art|arts|inc|par|pars|n|no|nos|sr|sra|dr|dra|prof|fig|tab|pag|ex|etc|cap)\.$")


def frases(texto):
    partes, atual = [], ""
    for pedaco in RE_FRASE.split(texto):
        pedaco = (pedaco or "").strip()
        if not pedaco:
            continue
        atual = f"{atual} {pedaco}".strip() if atual else pedaco
        if not ABREV.search(atual):
            partes.append(atual)
            atual = ""
    if atual:
        partes.append(atual)
    return partes


def quebrar_frase_longa(frase, maximo):
    """So para a frase que sozinha ja passa do limite: cortamos na pontuacao fraca."""
    pedacos, atual = [], ""
    for p in re.split(r"(?<=[;:,])\s+", frase):
        teste = f"{atual} {p}".strip()
        if atual and contar_tokens(teste) > maximo:
            pedacos.append(atual)
            atual = p
        else:
            atual = teste
    if atual:
        pedacos.append(atual)
    return pedacos


def segmentar(blocos, minimo=TOKENS_MIN, maximo=TOKENS_MAX):
    """Recebe os blocos de UM documento e devolve os trechos ja no tamanho pedido.
    Blocos curtos (incisos, itens de lista) sao costurados com os vizinhos, que e
    o que faz o texto do documento voltar a ter sentido corrido."""
    trechos, buffer, locais = [], [], []
    for b in blocos:
        for f in frases(b["texto"]):
            for pedaco in ([f] if contar_tokens(f) <= maximo else quebrar_frase_longa(f, maximo)):
                buffer.append(pedaco)
                locais.append(b["local"])
                texto = " ".join(buffer)
                if contar_tokens(texto) >= minimo:
                    trechos.append({"texto": texto, "fonte": b["fonte"],
                                    "local": f"{locais[0]}-{locais[-1]}" if locais[0] != locais[-1] else locais[0],
                                    "tokens": contar_tokens(texto)})
                    buffer, locais = [], []
    # O resto do documento so vira trecho se tiver tamanho util. O limite e baixo
    # de proposito: documento curto (um alerta de SMS, por exemplo) tem direito a
    # entrar no corpus, desde que entre inteiro e com a fonte registrada.
    if buffer:
        texto = " ".join(buffer)
        n = contar_tokens(texto)
        if n >= minimo // 3:
            trechos.append({"texto": texto, "fonte": blocos[-1]["fonte"],
                            "local": f"{locais[0]}-{locais[-1]}" if locais[0] != locais[-1] else locais[0],
                            "tokens": n})
    return trechos


trechos = []
resumo_seg = []
for arq, blocos in documentos_anon.items():
    t = segmentar(blocos)
    trechos.extend(t)
    resumo_seg.append((arq[:38], len(t),
                       f"{min((x['tokens'] for x in t), default=0)}-{max((x['tokens'] for x in t), default=0)}",
                       round(sum(x["tokens"] for x in t) / len(t)) if t else 0))

titulo("Trechos gerados por documento")
tabela(resumo_seg, ["Documento", "Trechos", "Faixa de tokens", "Media"], [38, 8, 16, 7])

_dentro = sum(1 for t in trechos if TOKENS_MIN <= t["tokens"] <= TOKENS_MAX)
print(f"\nTotal: {len(trechos)} trechos. Dentro da faixa {TOKENS_MIN}-{TOKENS_MAX}: "
      f"{_dentro} ({_dentro / len(trechos):.0%}).")
print("Os que ficam abaixo sao a sobra final de cada documento, que preferimos manter "
      "a jogar fora: sao trechos inteiros e com fonte.")


Trechos gerados por documento
Documento                               Trechos   Faixa de tokens   Media  
--------------------------------------  --------  ----------------  -------
cartilha_pronaf_credito_rural.pdf       70        155-293           263    
manual_garantia_safra_MAPA.pdf          22        256-296           268    
modelo_contrato_ater_MDA.docx           15        121-374           277    
lei_10420_2002_garantia_safra.html      29        256-288           268    
avisos_oficiais_inmet.txt               6         257-311           278    
recado_garantia_safra.txt               1         276-276           276    

Total: 143 trechos. Dentro da faixa 256-512: 141 (99%).
Os que ficam abaixo sao a sobra final de cada documento, que preferimos manter a jogar fora: sao trechos inteiros e com fonte.


## 6. Corpus 1: texto puro

O mais direto dos três. São os trechos da segmentação, com a fonte colada em cada linha.
É este corpus que ensina ao modelo o vocabulário do tema: *CAF*, *UFPA*, *módulo fiscal*,
*custeio*, *investimento*, *cota do produtor*, *entidade emissora*, *ano-safra*.

In [17]:
# @title Etapa 7a - Corpus de texto puro (minimo 50 trechos)
# Aqui nao tem nenhuma transformacao escondida: e o texto limpo da segmentacao,
# com a fonte colada em cada linha. E este corpus que ensina o vocabulario do tema.


def chave_dedup(texto):
    """Chave para duplicata exata. Minusculas e sem pontuacao SO para comparar;
    o texto que vai para o arquivo continua inteiro."""
    return re.sub(r"[^\wà-ÿ]+", " ", texto.lower()).strip()


def remover_duplicatas(itens, campo):
    vistos, saida, removidos = set(), [], 0
    for it in itens:
        k = chave_dedup(it[campo] if isinstance(campo, str) else campo(it))
        if k in vistos:
            removidos += 1
            continue
        vistos.add(k)
        saida.append(it)
    return saida, removidos


corpus_texto = [{"text": t["texto"], "fonte": t["fonte"], "trecho": t["local"],
                 "tokens": t["tokens"]} for t in trechos]
corpus_texto, dup_texto = remover_duplicatas(corpus_texto, "text")

titulo("Corpus de texto puro")
print(f"Trechos: {len(corpus_texto)} (minimo exigido: 50)")
print(f"Duplicatas exatas removidas: {dup_texto}")
print(f"Tokens por trecho: media {sum(t['tokens'] for t in corpus_texto) / len(corpus_texto):.0f}, "
      f"minimo {min(t['tokens'] for t in corpus_texto)}, maximo {max(t['tokens'] for t in corpus_texto)}")
tabela(sorted(Counter(t["fonte"] for t in corpus_texto).items(), key=lambda x: -x[1]),
       ["Fonte", "Trechos"], [46, 8])


Corpus de texto puro
Trechos: 141 (minimo exigido: 50)
Duplicatas exatas removidas: 2
Tokens por trecho: media 267, minimo 121, maximo 374
Fonte                                           Trechos 
----------------------------------------------  --------
cartilha_pronaf_credito_rural.pdf               68      
lei_10420_2002_garantia_safra.html              29      
manual_garantia_safra_MAPA.pdf                  22      
modelo_contrato_ater_MDA.docx                   15      
avisos_oficiais_inmet.txt                       6       
recado_garantia_safra.txt                       1       


## 7. Corpus 2: perguntas e respostas

### 7.1 Como geramos as perguntas

Começamos escrevendo moldes de pergunta, como nas outras etapas. Depois lemos a cartilha do
Pronaf com mais atenção e vimos que ela já é escrita em pergunta e resposta, com dúvidas
reais de agricultor:

> *Se a gente mora em um município e trabalha em outro, não pode fazer o CAF?*
> *Aposentadoria também entra como renda da família?*
> *Renda de arrendamento também conta?*

Essas perguntas são melhores que as nossas, então a família principal do corpus de QA
passou a extrair esse diálogo direto do documento, sem molde nenhum.

Tivemos dois problemas com isso. O primeiro é que a pergunta às vezes cai no fim de um
trecho e a resposta no começo do seguinte, então essa família roda no documento inteiro em
vez de trecho a trecho. O segundo é que o cabeçalho da cartilha também é uma pergunta
("Como acessar Crédito Rural pelo Pronaf?") e aparece 31 vezes. Em vez de escrever o título
na mão, descartamos qualquer pergunta repetida três vezes ou mais.

### 7.2 As outras famílias

Para o que não vem em forma de diálogo, usamos moldes. Cada um foi feito a partir de um
padrão que aparece nestes documentos:

| Família | Padrão que ela explora |
|---|---|
| pergunta do documento | o diálogo já escrito na cartilha |
| regra da lei | `Art. 5º O benefício será pago em até 3 parcelas...` |
| tópico do manual | títulos que já são perguntas (`COMO RESOLVER CASOS DE...`) |
| requisito | `considera-se ... desde que ...` |
| aviso meteorológico | campos do boletim do INMET |
| orientação prática e item de procedimento | frases no imperativo e listas com ponto e vírgula |
| assunto do trecho | rede de segurança, quando o trecho não rende nada melhor |

A resposta vem sempre do texto do documento. O que escrevemos é só a moldura em volta
("Segundo a Lei 10.420/2002, ..."), então a resposta não tem como fugir da fonte.

Na tabela de saída dá para ver que as famílias de orientação prática e de item de
procedimento quase não disparam. É diferença de tipo de documento: cartilha de emergência
manda ("não atravesse", "desligue a chave geral"), cartilha de crédito explica ("vocês
precisam ter o CAF ativo"). Verbo no imperativo é raro aqui.

Uma família nós escrevemos e depois tiramos. A terceira pergunta da persona é "em que porta
eu bato?", então fizemos uma para pegar as frases que citam sindicato, Emater, banco ou
entidade emissora. Ela rendeu só 3 pares úteis em 20 frases candidatas: os documentos citam
esses nomes bastante, mas quase nunca no formato "procure X para fazer Y". Achamos melhor
tirar do que deixar rendendo três pares. Ficou o registro de que a dúvida mais importante da
persona é a que estas fontes menos respondem.

### 7.3 Verificador de fidelidade (passo nosso)

O enunciado diz que as respostas não podem inventar informação. Para conferir isso, todo par
passa por duas checagens antes de entrar no corpus:

1. Todo número da resposta aparece no trecho de origem? Num corpus sobre benefício público,
   trocar "50% de perda" por "15%" muda quem tem direito.
2. As palavras de conteúdo da resposta vêm do trecho? Exigimos 80% de cobertura.

Uma observação de método: aqui usamos lista de palavras vazias, e isso não contradiz o
enunciado. Ele proíbe remover stop words do corpus. Na hora de comparar duas frases,
ignorar "de", "o" e "que" é o que faz a conta funcionar.

### 7.4 E se um LLM escrevesse as perguntas? (experimento)

O enunciado permite usar IA para criar os pares, então em vez de só opinar resolvemos medir.
Geramos pares com o Qwen2.5-1.5B-Instruct (licença Apache 2.0) e passamos todos pelo mesmo
verificador. O modelo e o prompt estão na célula abaixo e no relatório, como o enunciado
pede.

O verificador tem um limite que vale anotar: ele foi feito para resposta extraída, então
reprova também paráfrase correta. É conservador de propósito, porque numa resposta sobre
direito a benefício errar é pior do que não responder.

Sobre a revisão: o enunciado pede revisar pelo menos 20% dos exemplos quando se usa IA. A
verificação automática cobre 100%, mas ela não substitui leitura, então a célula seguinte
imprime a amostra de 25% que lemos, com o veredito do verificador ao lado de cada par.

In [18]:
# @title Etapa 7b - Perguntas e respostas a partir da estrutura dos documentos
# Decisao de projeto: a RESPOSTA sai sempre do texto do documento, nunca de um
# modelo. O que escrevemos a mao e so a moldura ("Segundo a Lei 11.326/2006, ...").
# Assim fidelidade deixa de ser promessa e vira consequencia da construcao.
#
# E a familia principal aqui nem moldura tem. Ao ler a cartilha do Pronaf
# descobrimos que ela ja e escrita em pergunta e resposta, com as duvidas reais do
# agricultor ("Aposentadoria tambem entra como renda da familia?"). Essas perguntas
# sao melhores do que qualquer molde que a gente inventasse: foram escritas pelo
# ministerio, na linguagem de quem pergunta.

APELIDO_FONTE = {
    "cartilha_pronaf_credito_rural.pdf": ("a", "cartilha do MDA sobre como acessar o credito rural pelo Pronaf"),
    "manual_garantia_safra_MAPA.pdf": ("o", "manual de orientacoes do Garantia-Safra"),
    "modelo_contrato_ater_MDA.docx": ("o", "modelo de contrato de ATER do MDA"),
    "lei_10420_2002_garantia_safra.html": ("a", "Lei 10.420/2002, que criou o Garantia-Safra"),
    "avisos_oficiais_inmet.txt": ("o", "boletim de avisos do INMET"),
    "recado_garantia_safra.txt": ("o", "recado sobre a adesao ao Garantia-Safra"),
}
CONTRAI_EM = {"a": "na", "o": "no", "as": "nas", "os": "nos"}
CONTRAI_DE = {"a": "da", "o": "do", "as": "das", "os": "dos"}


def citar(fonte, prep=None):
    """Monta a citacao ja com a preposicao certa. Sem isso sai 'Em o manual'."""
    art, nome = APELIDO_FONTE.get(fonte, ("o", fonte))
    if prep == "em":
        return f"{CONTRAI_EM[art]} {nome}"
    if prep == "de":
        return f"{CONTRAI_DE[art]} {nome}"
    return f"{art} {nome}"


def maiuscula(s):
    """Primeira letra maiuscula sem estragar o resto (capitalize() derruba siglas)."""
    return s[:1].upper() + s[1:]


def fechar(frase):
    frase = re.sub(r"\s+", " ", frase).strip(" ;,•-")
    return frase if frase.endswith((".", "!", "?")) else frase + "."


# Restos de extracao em coluna costumam colar duas preposicoes seguidas
# ("Corpo de do territorio"). Nenhuma frase correta em portugues faz isso.
RE_PREP_DUPLA = re.compile(
    r"(?i)(?<![\wà-ÿ])(de|do|da|dos|das|em|no|na|nos|nas|para|por|com)"
    r"\s+(de|do|da|dos|das|em|no|na|nos|nas)(?![\wà-ÿ])")


def frase_util(f, min_palavras=7):
    """Descarta frase que o extrator picou: item de lista solto, frase que termina
    em numero de topico, parentese aberto, bullet no meio."""
    f = f.strip()
    if len(f.split()) < min_palavras or f.count("•") > 0:
        return False
    if re.search(r"\b\d\.$", f) or f.count("(") != f.count(")"):
        return False
    if RE_PREP_DUPLA.search(f):
        return False
    # Fragmento de titulo em caixa alta ("INDICIO: VINCULO COM A TERRA OU CULTIVO EM
    # AREA MAIOR QUE O PERMITIDO"). O manual do Garantia-Safra e cheio deles, e eles
    # passavam por todos os outros filtros porque sao frases bem formadas.
    palavras_alfa = re.findall(r"[A-Za-zÀ-ÿ]{3,}", f)
    if palavras_alfa:
        caixa_alta = sum(1 for p in palavras_alfa if p.isupper())
        if caixa_alta / len(palavras_alfa) > 0.35:
            return False
    return bool(re.search(r"[a-zà-ÿ]{4}", f))


def novo_par(destino, pergunta, resposta, trecho, familia):
    destino.append({"instruction": re.sub(r"\s+", " ", pergunta).strip(),
                    "input": "", "output": fechar(resposta),
                    "fonte": trecho["fonte"], "trecho": trecho["local"],
                    "contexto": trecho["texto"], "familia": familia})


# --- F1: a pergunta ja esta no documento ------------------------------------
# A cartilha do Pronaf e um dialogo. Em vez de tentar casar pergunta e resposta
# com uma unica expressao regular (que perdia dois tercos dos pares), localizamos
# TODAS as perguntas e tratamos como resposta o texto que vai ate a proxima.
# Localizar a pergunta por uma unica expressao nao funciona bem: proibir o ponto
# dentro dela perde um terco dos pares (abreviacao, "R$ 1.500,00"), e permitir faz
# ela engolir as frases anteriores ate o "?". Entao achamos o "?" e caminhamos para
# tras ate o fim da frase anterior.
RE_INTERROGACAO = re.compile(r"\?")
RE_FIM_DE_FRASE = re.compile(r"[.!?]\s+(?=[A-ZÀ-Þ])|\n")


def localizar_perguntas(texto):
    """Devolve (inicio, fim) de cada pergunta do texto."""
    cortes = [0] + [m.end() for m in RE_FIM_DE_FRASE.finditer(texto)]
    achadas = []
    for m in RE_INTERROGACAO.finditer(texto):
        inicio = max((c for c in cortes if c <= m.start()), default=0)
        bruto = texto[inicio:m.end()]
        # a pergunta tem que comecar com maiuscula e ter tamanho de pergunta
        deslocamento = len(bruto) - len(bruto.lstrip())
        bruto = bruto.strip()
        if 16 <= len(bruto) <= 170 and re.match(r"[A-ZÀ-Þ]", bruto):
            achadas.append((inicio + deslocamento, m.end()))
    return achadas


MAX_RESPOSTA = 620

# --- F2: definicoes dos incisos da lei --------------------------------------
RE_DEF_LEI = re.compile(r"(?:^|\s)[IVXL]{1,6}\s*[-–]\s*([a-zà-ÿ][\wà-ÿ\s]{2,45}?):\s+([a-zà-ÿ][^;]{40,400}?)\s*[;.]")
MOLDES_DEF = ["O que significa {t} na lei da agricultura familiar?",
              "O que e considerado {t}?",
              "Como a lei define {t}?",
              "O que quer dizer {t} nesse contexto?"]

# --- F3: requisitos ---------------------------------------------------------
# "considera-se agricultor familiar ... desde que ..." e o coracao da Lei 11.326:
# e a resposta para "eu me encaixo ou nao?", que e a primeira duvida da persona.
RE_REQUISITO = re.compile(
    r"(?i)(considera-se\s+[\wà-ÿ\s]{4,60}?|para\s+(?:ter|acessar|aderir|participar)[\wà-ÿ\s]{0,40}?)"
    r"[,\s]+(?:e\s+preciso|desde que|e necessario|é necessário)\s+([^.]{40,420})\.")
MOLDES_REQ = ["Quais sao os requisitos para {t}?",
              "O que eu preciso cumprir para {t}?",
              "Quem tem direito a {t}?"]

# --- F4: orientacoes praticas (frases no imperativo) ------------------------
# "Nao" ficou de fora de proposito: sozinho ele casa com qualquer frase negativa
# ("Nao faz diferenca se o pagamento..."), que nao e orientacao nenhuma.
VERBOS_IMP = ("Evite", "Mantenha", "Procure", "Nunca", "Leve", "Guarde",
              "Observe", "Acompanhe", "Verifique", "Prepare", "Tenha", "Peca", "Peça",
              "Apresente", "Solicite", "Informe", "Confira", "Consulte", "Compareca",
              "Compareça", "Atualize", "Comprove", "Preencha", "Assine", "Pague",
              "Cadastre", "Entregue", "Lembre", "Fique", "Avise", "Separe", "Reuna",
              "Reúna", "Faca", "Faça", "Use", "Escolha", "Combine", "Planeje")
RE_IMPERATIVO = re.compile(r"(?:^|(?<=[.!?;:])\s)((?:" + "|".join(VERBOS_IMP) + r")\b[^.!?;•]{25,230}[.!?])")
ASSUNTOS = [("garantia-safra", "o Garantia-Safra"), ("garantia safra", "o Garantia-Safra"),
            ("caf", "o CAF"), ("dap", "a DAP"), ("pronaf", "o Pronaf"),
            ("ater", "a assistencia tecnica"), ("seca", "a seca"), ("estiagem", "a estiagem"),
            ("geada", "a geada"), ("granizo", "o granizo"),
            ("credito", "o credito rural"), ("crédito", "o credito rural")]
MOLDES_ORI = ["O que eu devo fazer quando o assunto e {a}?",
              "Qual e a orientacao oficial sobre {a}?",
              "Como eu me preparo para {a}?"]

# --- F5: itens de procedimento separados por ponto e virgula ----------------
RE_ITEM_LISTA = re.compile(r"(?:^|[;:•])\s*((?:" + "|".join(VERBOS_IMP) + r")\b[^;:•.!?]{20,200})")
MOLDES_ITEM = ["Tem alguma recomendacao sobre {a}?",
               "O que a orientacao oficial diz para quem precisa d{a2}?",
               "Qual cuidado eu preciso ter com {a}?"]

# --- F6: avisos meteorologicos do INMET -------------------------------------
RE_AVISO = re.compile(r"AVISO:\s*([^-]{3,40}?)\s*-\s*Severidade:\s*([A-ZÀ-Þ][^.]{3,40}?)\s+(?:Vigencia|Estados|Riscos)")
RE_RISCO = re.compile(r"Riscos previstos:\s*(.{40,400}?)(?=\s+Orientacao a populacao:|\s+AVISO:|\s+Vigencia:|\s+Estados sob|$)")
RE_ORIENTA = re.compile(r"Orientacao a populacao:\s*(.{25,300}?)(?=\s+Orientacao a populacao:|\s+AVISO:|\s+Vigencia:|\s+Riscos previstos:|\s+Estados sob|$)")

# --- F6b: a regra que esta no artigo da lei ---------------------------------
# A Lei 10.420 nao usa o formato "inciso: definicao", entao a familia de definicao
# nao pega nada nela. O que ela tem sao artigos com a regra do programa, e e isso
# que responde "o beneficio e pago em quantas parcelas?".
RE_ARTIGO = re.compile(r"(?:Art\.\s*\d+[ºo]?[-A]*|§\s*\d+[ºo]?|Par[áa]grafo [úu]nico)\s*\.?\s*"
                       r"([A-ZÀ-Þ][^.]{60,340}\.)")
MOLDES_LEI = ["O que a lei diz sobre {a}?",
              "Qual e a regra oficial sobre {a}?",
              "O que esta previsto em lei a respeito d{a2}?"]

# --- F6c: titulo em caixa alta do manual ------------------------------------
# O manual do Garantia-Safra organiza o conteudo em titulos que ja sao perguntas
# disfarcadas ("COMO RESOLVER CASOS DE INDICIOS DE DESCONFORMIDADE").
RE_TITULO_CAIXA = re.compile(r"((?:COMO|QUANDO|QUEM|ONDE|O QUE|QUAIS)[A-ZÀ-Þ0-9\s,\-]{10,70})\s+"
                             r"([A-ZÀ-Þ][^.]{50,340}\.)")

# --- F7: "onde resolver" foi tentada e descartada ---------------------------
# A terceira pergunta da persona e "em que porta eu bato?", entao escrevemos uma
# familia para extrair as frases que citam sindicato, Emater, banco ou entidade
# emissora. Ela rendeu 3 pares uteis em 20 frases candidatas: os documentos citam
# esses nomes o tempo todo, mas quase nunca na forma "procure X para fazer Y".
# Tiramos a familia em vez de deixa-la rendendo tres pares e um nome bonito na
# tabela. O achado em si esta no relatorio: a pergunta mais importante da persona
# e justamente a que estas fontes menos respondem de forma direta.


# --- F8: o assunto do trecho ------------------------------------------------
TERMOS_TEMA = [
    "pronaf", "garantia-safra", "garantia safra", "caf", "dap", "credito rural",
    "crédito rural", "ater", "assistencia tecnica", "assistência técnica",
    "extensao rural", "extensão rural", "agricultor familiar", "agricultura familiar",
    "modulo fiscal", "módulo fiscal", "custeio", "investimento", "juros", "safra",
    "lavoura", "seca", "estiagem", "cota", "beneficio", "benefício", "sindicato",
    "cooperativa", "projeto tecnico", "projeto técnico", "renda", "area", "área",
    "financiamento", "parcela", "contrato", "vistoria", "perda", "producao", "produção",
]
MOLDES_TEMA = ["O que os documentos oficiais falam sobre {t}?",
               "Onde entra {t} no acesso as politicas da agricultura familiar?",
               "O que eu preciso saber sobre {t}?"]

RE_NUM_SECAO = re.compile(r"^\s*\d+(?:\.\d+)*\s+(?=[A-ZÀ-Þ])")
# Titulo de secao que o extrator colou no comeco da frase ("APRESENTACAO O
# Garantia-Safra, criado pela Lei..."). So tiramos palavras de 6 letras ou mais em
# caixa alta: assim "CAF e o cadastro..." e "DAP ativa" ficam intactos.
RE_TITULO_COLADO = re.compile(r"^(?:(?:[A-ZÀ-Þ]{6,}|DE|DA|DO|DOS|DAS|E|OU|COM|EM|NO|NA)\s+){1,6}"
                              r"(?=[A-ZÀ-Þ])")


def tirar_titulo_colado(frase):
    limpa = RE_TITULO_COLADO.sub("", frase, count=1)
    # so aceitamos a limpeza se ainda sobrar frase de verdade
    return limpa if len(limpa.split()) >= 8 else frase


def frases_com_termo(texto, termo, maximo=3):
    achadas = []
    for f in re.split(r"(?<=[.!?])\s+", texto):
        f = tirar_titulo_colado(RE_NUM_SECAO.sub("", f).strip()).strip()
        if not re.match(r"[A-ZÀ-Þ]", f) or re.match(r"[IVXL]{1,6}\s*[-–]", f):
            continue
        if termo in f.lower() and frase_util(f, 8):
            achadas.append(f)
    return achadas[:maximo]


def perguntas_do_documento(lista_trechos):
    """F1 roda no documento inteiro, nao trecho a trecho: a pergunta cai no fim de
    um trecho e a resposta no comeco do seguinte.

    O cabecalho corrente da cartilha tambem e uma pergunta ("Como acessar Credito
    Rural pelo Pronaf?") e aparece 31 vezes. Em vez de escrever o titulo na mao,
    descartamos qualquer pergunta que se repita: cabecalho repete, duvida de
    agricultor nao."""
    pares = []
    por_fonte = defaultdict(list)
    for t in lista_trechos:
        por_fonte[t["fonte"]].append(t)

    for fonte, trechos_f in por_fonte.items():
        inteiro = " ".join(t["texto"] for t in trechos_f)
        achadas = localizar_perguntas(inteiro)
        frequencia = Counter(" ".join(inteiro[a:b].split()) for a, b in achadas)

        for i, (ini, fim) in enumerate(achadas):
            pergunta = " ".join(inteiro[ini:fim].split())
            # Descartamos tambem a pergunta que carrega pedaco de titulo em caixa
            # alta ("PASSO: FAZER O CAF para que ele serve?"): e cabecalho colado na
            # frase seguinte, nao duvida de ninguem.
            if (frequencia[pergunta] >= 3                       # cabecalho corrente
                    or pergunta.isupper() or len(pergunta.split()) < 4
                    or not re.search(r"[a-zà-ÿ]{3}", pergunta)
                    or re.search(r"\b[A-ZÀ-Þ]{2,}\s+[A-ZÀ-Þ]{2,}\b", pergunta)
                    or ":" in pergunta or ".indd" in pergunta):
                continue
            # A resposta vai do fim desta pergunta ate o inicio da proxima.
            limite = achadas[i + 1][0] if i + 1 < len(achadas) else len(inteiro)
            bruto = inteiro[fim:min(limite, fim + MAX_RESPOSTA)].strip()
            # Fecha na ultima frase completa, para nao entregar resposta cortada.
            corte = max(bruto.rfind("."), bruto.rfind("!"))
            if corte < 40:
                continue
            resposta = " ".join(bruto[:corte + 1].split())
            if (".indd" in resposta or re.match(r"(?:[A-ZÀ-Þ]{2,}\s+){2,}", resposta)
                    or not frase_util(resposta, 8)):
                continue
            ini_c, fim_c = max(0, ini - 600), min(len(inteiro), fim + 900)
            dono = next((t for t in trechos_f if pergunta[:40] in t["texto"]), trechos_f[0])
            pares.append({"instruction": pergunta, "input": "", "output": fechar(resposta),
                          "fonte": fonte, "trecho": dono["local"],
                          "contexto": inteiro[ini_c:fim_c], "familia": "pergunta_do_documento"})
    return pares


def gerar_qa_por_regra(lista_trechos):
    """Aplica todos os moldes acima a uma lista de trechos. A interface usa esta
    mesma funcao, para que notebook e ferramenta nao se afastem com o tempo."""
    pares = perguntas_do_documento(lista_trechos)
    _ordem = Counter()
    for t in lista_trechos:
        ctx, fonte = t["texto"], t["fonte"]

        for m in RE_DEF_LEI.finditer(ctx):
            termo, definicao = m.group(1).strip().lower(), m.group(2).strip()
            if len(termo.split()) > 5 or termo in ("vetado",) or not frase_util(definicao, 8):
                continue
            _ordem["def"] += 1
            novo_par(pares, MOLDES_DEF[_ordem["def"] % len(MOLDES_DEF)].format(t=termo),
                     f"Segundo {citar(fonte)}, {termo} e {definicao}", t, "definicao_legal")

        for m in RE_REQUISITO.finditer(ctx):
            sujeito = m.group(1).strip().strip(" ,")
            sujeito = re.sub(r"(?i)^considera-se\s+", "ser considerado ", sujeito)
            sujeito = re.sub(r"(?i)^para\s+", "", sujeito).strip()
            exigencia = m.group(2).strip()
            if len(sujeito.split()) > 8 or not frase_util(exigencia, 8):
                continue
            _ordem["req"] += 1
            novo_par(pares, MOLDES_REQ[_ordem["req"] % len(MOLDES_REQ)].format(t=sujeito),
                     f"De acordo com {citar(fonte)}, e preciso {exigencia}", t, "requisito")

        for m in RE_IMPERATIVO.finditer(ctx):
            ordem = re.sub(r"\s+", " ", m.group(1)).strip()
            if not frase_util(ordem, 7):
                continue
            assunto = next((rot for ch, rot in ASSUNTOS if ch in ctx.lower()), "o credito rural")
            _ordem["ori"] += 1
            novo_par(pares, MOLDES_ORI[_ordem["ori"] % len(MOLDES_ORI)].format(a=assunto),
                     f"Conforme {citar(fonte)}: {ordem}", t, "orientacao_pratica")

        assunto = next((rot for ch, rot in ASSUNTOS if ch in ctx.lower()), "o credito rural")
        for m in RE_ITEM_LISTA.finditer(ctx):
            item = re.sub(r"\s+", " ", m.group(1)).strip(" ;:.")
            if not frase_util(item, 7):
                continue
            _ordem["item"] += 1
            molde = MOLDES_ITEM[_ordem["item"] % len(MOLDES_ITEM)]
            novo_par(pares, molde.format(a=assunto, a2=assunto[1:]),
                     f"Conforme {citar(fonte)}: {item}", t, "item_de_procedimento")

        for m in RE_AVISO.finditer(ctx):
            tipo, sev = m.group(1).strip(), m.group(2).strip()
            risco = RE_RISCO.search(ctx[m.end():m.end() + 900])
            if risco and frase_util(risco.group(1), 6):
                novo_par(pares, f"O que significa um aviso de {tipo} com severidade {sev}?",
                         f"Segundo {citar(fonte)}, nesse tipo de aviso os riscos previstos sao: "
                         f"{risco.group(1)}", t, "aviso_meteorologico")

        for m in RE_ORIENTA.finditer(ctx):
            orienta = re.sub(r"\s+", " ", m.group(1)).strip()
            if frase_util(orienta, 6):
                novo_par(pares, "O que o INMET orienta a fazer quando sai um aviso meteorologico?",
                         f"Conforme {citar(fonte)}: {orienta}", t, "aviso_meteorologico")


        for m in RE_ARTIGO.finditer(ctx):
            regra = re.sub(r"\s+", " ", m.group(1)).strip()
            if not frase_util(regra, 10):
                continue
            _ordem["lei"] += 1
            molde = MOLDES_LEI[_ordem["lei"] % len(MOLDES_LEI)]
            novo_par(pares, molde.format(a=assunto, a2=assunto[1:]),
                     f"Segundo {citar(fonte)}: {regra}", t, "regra_da_lei")

        for m in RE_TITULO_CAIXA.finditer(ctx):
            titulo_bruto = " ".join(m.group(1).split()).strip(" ,-")
            conteudo = re.sub(r"\s+", " ", m.group(2)).strip()
            # Titulo cortado no meio termina em preposicao ou artigo ("...com a lei
            # do?"), e isso nao e pergunta.
            if (len(titulo_bruto.split()) < 4 or not frase_util(conteudo, 10)
                    or titulo_bruto.split()[-1].lower() in
                    ("do", "da", "de", "dos", "das", "com", "em", "no", "na", "o", "a",
                     "os", "as", "ao", "aos", "e", "ou", "para", "por", "que")):
                continue
            pergunta = maiuscula(titulo_bruto.lower()) + "?"
            novo_par(pares, pergunta,
                     f"{maiuscula(citar(fonte, 'em'))} aparece o seguinte: {conteudo}",
                     t, "topico_do_manual")

        # F8: os assuntos do trecho, no maximo dois e so quando trazem frases diferentes.
        # Esta familia e a rede de seguranca. Se o trecho ja rendeu pares pelas
        # familias boas, nao precisamos dela ali: so engrossaria o corpus com
        # pergunta generica.
        ja_rendeu = sum(1 for p in pares if p["contexto"] is ctx)
        # Quanto menos o trecho rendeu pelas familias boas, mais espaco a rede de
        # seguranca ganha nele. Se ja rendeu dois pares, ela nao entra.
        teto_tema = 2 if ja_rendeu == 0 else (1 if ja_rendeu == 1 else 0)
        contagem = sorted(((ctx.lower().count(tm), tm) for tm in TERMOS_TEMA), reverse=True)
        ja_usadas = set()
        for quantas, termo in contagem[:4]:
            if quantas < 2 or len(ja_usadas) >= teto_tema:
                continue
            frases_t = frases_com_termo(ctx, termo)
            chave = chave_dedup(" ".join(frases_t))
            if not frases_t or chave in ja_usadas:
                continue
            ja_usadas.add(chave)
            _ordem["tema"] += 1
            novo_par(pares, MOLDES_TEMA[_ordem["tema"] % len(MOLDES_TEMA)].format(t=termo),
                     f"{maiuscula(citar(fonte, 'em'))} aparece o seguinte: " + " ".join(frases_t),
                     t, "assunto_do_trecho")

    return pares


qa_regras = gerar_qa_por_regra(trechos)

titulo("Pares gerados por regra, antes de filtrar")
tabela(sorted(Counter(p["familia"] for p in qa_regras).items(), key=lambda x: -x[1]),
       ["Familia de pergunta", "Pares"], [26, 8])
print(f"\nTotal bruto: {len(qa_regras)} pares.")


Pares gerados por regra, antes de filtrar
Familia de pergunta         Pares   
--------------------------  --------
assunto_do_trecho           88      
pergunta_do_documento       23      
aviso_meteorologico         13      
regra_da_lei                10      
topico_do_manual            3       
requisito                   1       
orientacao_pratica          1       

Total bruto: 139 pares.


In [19]:
# @title Verificador de fidelidade (passo nosso)
# O enunciado cobra que "as respostas nao podem inventar informacoes". Cobrar e
# facil; conferir e que e o trabalho. Este verificador roda em CADA par antes de
# ele entrar no corpus e pergunta duas coisas sobre a resposta:
#
#   1) Todo numero que aparece na resposta aparece no trecho de origem?
#      Numero inventado e o erro mais caro aqui: trocar "50% de perda" por "15%" ou
#      "3 parcelas" por "30" muda quem tem direito e quanto recebe.
#   2) As palavras de conteudo da resposta vem do trecho?
#
# Repare que AQUI usamos lista de palavras vazias. Isso nao contradiz o enunciado:
# ele proibe remover stop words DO CORPUS. Para comparar duas frases, ignorar
# "de", "o", "que" e justamente o que faz a medida significar alguma coisa.

VAZIAS = set("""a o e as os da de do das dos na no nas nos um uma uns umas que se ao aos
a as por para com sem sob sobre entre ate apos antes quando onde como qual quais e ou
mas nem seu sua seus suas este esta isso aquele aquela ser estar ter haver foi sao e
em pelo pela pelos pelas numa num deve devem pode podem ja nao sim mais menos muito
segundo conforme acordo caso cada todo toda todos todas outro outra qualquer""".split())
RE_NUMERO = re.compile(r"\d+(?:[.,]\d+)?")


def _conteudo(texto):
    palavras = re.findall(r"[\wà-ÿ]{3,}", texto.lower())
    return [p for p in palavras if p not in VAZIAS]


# A moldura que nos escrevemos cita "Lei 10.420/2002". Esses numeros sao nossos e
# nao estao no trecho: na primeira rodada o verificador reprovou todas as citacoes
# da lei por causa deles. Como a moldura e conhecida, ela entra na base de
# comparacao em vez de ser tratada como invencao.
MOLDURA = " ".join(nome for _, nome in APELIDO_FONTE.values()) + \
    " segundo conforme de acordo com procure aparece assim aparece o seguinte e preciso " \
    "nesse tipo de aviso os riscos previstos sao lei fundo programa garantia safra"
NUMEROS_MOLDURA = set(RE_NUMERO.findall(MOLDURA))
PALAVRAS_MOLDURA = set(_conteudo(MOLDURA))


def verificar_fidelidade(resposta, contexto, minimo=0.80):
    """Devolve (aprovado, cobertura, numeros_inventados)."""
    nums_ctx = set(RE_NUMERO.findall(contexto)) | NUMEROS_MOLDURA
    inventados = [n for n in RE_NUMERO.findall(resposta) if n not in nums_ctx]

    alvo = _conteudo(resposta)
    if not alvo:
        return False, 0.0, inventados
    base = set(_conteudo(contexto)) | PALAVRAS_MOLDURA
    cobertura = sum(1 for p in alvo if p in base) / len(alvo)
    return (cobertura >= minimo and not inventados), cobertura, inventados


# Rodando sobre os pares gerados por regra. Como a resposta e extraida do trecho,
# a expectativa e que quase tudo passe: se nao passar, o erro esta no nosso gerador.
_aprov, _reprov, _cobs = [], [], []
for p in qa_regras:
    ok, cob, inv = verificar_fidelidade(p["output"], p["contexto"])
    _cobs.append(cob)
    p["cobertura"] = round(cob, 3)
    (_aprov if ok else _reprov).append((p, cob, inv))

qa_regras = [p for p, _, _ in _aprov]

titulo("Verificacao de fidelidade dos pares gerados por regra")
print(f"Aprovados: {len(_aprov)} de {len(_cobs)}  |  reprovados: {len(_reprov)}")
COBERTURA_REGRAS = sum(_cobs) / len(_cobs)
print(f"Cobertura media da resposta pelo trecho de origem: {COBERTURA_REGRAS:.1%}")
if _reprov:
    print("\nO que foi reprovado (e por que o nosso gerador errou):")
    for p, cob, inv in _reprov[:4]:
        motivo = f"numeros fora do trecho: {inv}" if inv else f"cobertura {cob:.0%} abaixo de 80%"
        print(f"  [{p['familia']}] {motivo}")
        print(f"     P: {p['instruction']}")
        print(f"     R: {amostra(p['output'], 150)}")


Verificacao de fidelidade dos pares gerados por regra
Aprovados: 139 de 139  |  reprovados: 0
Cobertura media da resposta pelo trecho de origem: 100.0%


In [20]:
# @title Experimento: e se as perguntas fossem geradas por um LLM?
# O enunciado permite usar IA para criar os pares. Nos usamos regra para o corpus
# principal, mas achamos que valia medir a alternativa em vez de so opinar sobre
# ela. Entao geramos pares com um modelo pequeno e passamos TUDO pelo mesmo
# verificador de fidelidade. O que passa entra no corpus; o que nao passa vira
# estatistica no relatorio.
#
# Modelo usado: Qwen2.5-1.5B-Instruct (licenca Apache 2.0, roda no Colab gratuito).
# O prompt esta na variavel PROMPT_QA abaixo, reproduzido no relatorio.

MODELO_QA = "Qwen/Qwen2.5-1.5B-Instruct"
N_TRECHOS_LLM = 12      # calibrado para rodar em CPU; com GPU da para subir bastante
USAR_LLM = True         # desligue para rodar o notebook inteiro sem baixar o modelo

PROMPT_QA = """Voce recebe um trecho de um documento oficial brasileiro sobre as politicas
publicas da agricultura familiar (CAF, Pronaf, Garantia-Safra e assistencia tecnica).

Escreva 2 perguntas que uma agricultora familiar faria sobre este trecho, e a
resposta de cada uma. Regras:
- A resposta so pode usar informacao que esta no trecho. Nao acrescente nada.
- Escreva em portugues simples, como quem explica para alguem que nao e tecnico.
- Se o trecho nao responde nada util para uma agricultora, devolva apenas NADA.

Responda exatamente neste formato, sem comentarios:
P: <pergunta 1>
R: <resposta 1>
P: <pergunta 2>
R: <resposta 2>

Trecho:
\"\"\"{trecho}\"\"\""""

qa_llm, _brutos_llm = [], []
_tempo_llm = 0.0

if USAR_LLM:
    import time as _time
    try:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer as _AT

        _dev = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Carregando {MODELO_QA} em {_dev}...")
        _tok = _AT.from_pretrained(MODELO_QA)
        _mod = AutoModelForCausalLM.from_pretrained(
            MODELO_QA, torch_dtype=torch.float16 if _dev == "cuda" else torch.float32).to(_dev)
        _mod.eval()

        _sel = random.Random(SEED).sample(trechos, min(N_TRECHOS_LLM, len(trechos)))
        _t0 = _time.time()
        for _i, _t in enumerate(_sel, 1):
            _msgs = [{"role": "user", "content": PROMPT_QA.format(trecho=_t["texto"][:1800])}]
            _entrada = _tok.apply_chat_template(_msgs, tokenize=False, add_generation_prompt=True)
            _ids = _tok(_entrada, return_tensors="pt").to(_dev)
            with torch.no_grad():
                _saida = _mod.generate(**_ids, max_new_tokens=220, do_sample=False,
                                       pad_token_id=_tok.eos_token_id)
            _txt = _tok.decode(_saida[0][_ids["input_ids"].shape[1]:], skip_special_tokens=True)
            for _m in re.finditer(r"P:\s*(.+?)\s*\n+R:\s*(.+?)(?=\n+P:|\Z)", _txt, re.S):
                _p, _r = _m.group(1).strip(), re.sub(r"\s+", " ", _m.group(2)).strip()
                if len(_p) > 10 and len(_r) > 20:
                    _brutos_llm.append({"instruction": _p, "input": "", "output": _r,
                                        "fonte": _t["fonte"], "trecho": _t["local"],
                                        "contexto": _t["texto"], "familia": "gerado_por_llm"})
            if _i % 3 == 0:
                print(f"  {_i}/{len(_sel)} trechos processados "
                      f"({(_time.time() - _t0) / _i:.0f}s por trecho)")
        _tempo_llm = _time.time() - _t0
        print(f"Geracao concluida em {_tempo_llm:.0f}s.")
    except Exception as _e:
        print(f"(aviso) nao foi possivel rodar o LLM: {type(_e).__name__}: {_e}")
        print("O corpus principal nao depende desta celula; ela e um experimento.")

# --- O verificador agora tem trabalho de verdade ---------------------------
if _brutos_llm:
    _ok, _rej = [], []
    for _p in _brutos_llm:
        _aprovado, _cob, _inv = verificar_fidelidade(_p["output"], _p["contexto"])
        _p["cobertura"] = round(_cob, 3)
        _p["numeros_fora"] = _inv
        (_ok if _aprovado else _rej).append(_p)
    qa_llm = _ok

    titulo("Pares gerados por LLM: quanto sobrevive ao verificador de fidelidade")
    print(f"Gerados: {len(_brutos_llm)} a partir de {min(N_TRECHOS_LLM, len(trechos))} trechos")
    print(f"Aprovados: {len(_ok)} ({len(_ok) / len(_brutos_llm):.0%})  |  "
          f"Reprovados: {len(_rej)} ({len(_rej) / len(_brutos_llm):.0%})")
    print(f"Cobertura media: LLM {sum(p['cobertura'] for p in _brutos_llm) / len(_brutos_llm):.1%} "
          f"contra {COBERTURA_REGRAS:.1%} das respostas extraidas por regra.")
    _com_numero = [p for p in _rej if p["numeros_fora"]]
    print(f"Reprovados por numero que nao existe no trecho: {len(_com_numero)}")

    if _rej:
        titulo("Exemplos reprovados (e o que o modelo inventou)", "-")
        for _p in _rej[:3]:
            _motivo = (f"numeros fora do trecho: {_p['numeros_fora']}" if _p["numeros_fora"]
                       else f"cobertura {_p['cobertura']:.0%}")
            print(f"\n  [{_motivo}]")
            print(f"  P: {amostra(_p['instruction'], 120)}")
            print(f"  R: {amostra(_p['output'], 230)}")
else:
    print("Nenhum par gerado por LLM nesta execucao.")

Carregando Qwen/Qwen2.5-1.5B-Instruct em cpu...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/338 [00:01<06:03,  1.08s/it]

Loading weights:  26%|██▌       | 87/338 [00:01<00:02, 100.70it/s]

Loading weights:  40%|████      | 136/338 [00:01<00:01, 112.17it/s]

... (14 outputs left out)

  3/12 trechos processados (42s por trecho)


  6/12 trechos processados (35s por trecho)


  9/12 trechos processados (33s por trecho)


  12/12 trechos processados (36s por trecho)
Geracao concluida em 432s.

Pares gerados por LLM: quanto sobrevive ao verificador de fidelidade
Gerados: 24 a partir de 12 trechos
Aprovados: 1 (4%)  |  Reprovados: 23 (96%)
Cobertura media: LLM 39.3% contra 100.0% das respostas extraidas por regra.
Reprovados por numero que nao existe no trecho: 0

------------------------------------------------------------------------------
Exemplos reprovados (e o que o modelo inventou)
------------------------------------------------------------------------------

  [cobertura 42%]
  P: Como faço para obter crédito através do Pronaf?
  R: Para obter crédito através do Pronaf, você precisa assinar alguns papéis e seguir um processo
burocrático. Além disso, é necessário apresentar documentos como declaração negativa de débitos com
o INSS e comprovar os preços das sua

  [cobertura 32%]
  P: Como posso garantir meu acesso ao crédito através do Pronaf?
  R: Para garantir seu acesso ao crédito através do Pr

In [21]:
# @title Lista para a revisao manual dos pares gerados por IA
# O enunciado pede que, se usarmos IA, o grupo revise pelo menos 20% dos exemplos e
# informe o modelo e o prompt. A verificacao automatica da celula anterior cobre
# 100%, mas verificacao automatica nao e leitura humana. Esta celula imprime a
# amostra que lemos, com o veredito do verificador ao lado de cada par, para que a
# revisao fique registrada e qualquer pessoa possa refazer a mesma conferencia.

FRACAO_REVISAO = 0.25   # acima dos 20% exigidos

if _brutos_llm:
    _n = max(1, round(FRACAO_REVISAO * len(_brutos_llm)))
    _amostra = random.Random(SEED).sample(_brutos_llm, _n)
    titulo(f"Amostra para revisao manual: {_n} de {len(_brutos_llm)} pares "
           f"({_n / len(_brutos_llm):.0%}; minimo exigido 20%)")
    print(f"Modelo: {MODELO_QA}")
    print("Prompt: variavel PROMPT_QA desta secao, reproduzido no relatorio.\n")
    for _i, _p in enumerate(_amostra, 1):
        _ok, _cob, _inv = verificar_fidelidade(_p["output"], _p["contexto"])
        _veredito = "APROVADO" if _ok else (
            f"REPROVADO (numeros fora: {_inv})" if _inv else f"REPROVADO (cobertura {_cob:.0%})")
        print(f"[{_i:>2}] {_veredito}  |  fonte: {_p['fonte']} ({_p['trecho']})")
        print(f"     P: {amostra(_p['instruction'], 150)}")
        print(f"     R: {amostra(_p['output'], 300)}")
        print()
    print("O que encontramos lendo esta amostra esta descrito na secao 7.3 e no relatorio.")
else:
    titulo("Revisao manual de pares gerados por IA")
    print("Nenhum par foi gerado por IA nesta execucao (USAR_LLM esta desligado),")
    print("entao o corpus final nao contem exemplo de IA e a exigencia dos 20% nao se aplica.")


Amostra para revisao manual: 6 de 24 pares (25%; minimo exigido 20%)
Modelo: Qwen/Qwen2.5-1.5B-Instruct
Prompt: variavel PROMPT_QA desta secao, reproduzido no relatorio.

[ 1] REPROVADO (cobertura 40%)  |  fonte: cartilha_pronaf_credito_rural.pdf (p.12)
     P: Como posso solicitar o crédito rural através do Pronaf?
     R: Para solicitar o crédito rural através do Pronaf, é necessário estar próximo ao local onde
trabalhamos todos os dias. O importante é que possamos chegar lá facilmente, independentemente do
município onde vivemos. Além disso, aposentadorias e benefícios sociais também podem ser
consideradas como rend

[ 2] REPROVADO (cobertura 38%)  |  fonte: cartilha_pronaf_credito_rural.pdf (p.9-p.11)
     P: Como posso obter o crédito rural através do Pronaf?
     R: Você deve ter morado na propriedade ou em um local próximo dela, com uma distância suficiente para
poder chegar ao trabalho diariamente. A área da propriedade não pode ser muito grande, podendo
atingir até 4 módulos 

In [22]:
# @title Etapa 7b - Montar o corpus de perguntas e respostas (minimo 100 pares)
# Juntamos os pares por regra com os pares do LLM que passaram no verificador.
# Duplicata aqui nao e so texto igual: e tambem pergunta igual com resposta igual,
# que aparece quando dois trechos repetem a mesma orientacao.

corpus_qa_bruto = qa_regras + qa_llm
corpus_qa, dup_qa = remover_duplicatas(
    corpus_qa_bruto, lambda p: p["instruction"] + " || " + p["output"])

# Mesma pergunta com respostas diferentes e saudavel ate certo ponto (ensina o
# modelo a responder de mais de um jeito), mas repetir demais desbalanceia o
# treino. Limitamos a 4 respostas por pergunta.
LIMITE_POR_PERGUNTA = 4
_por_pergunta, _corte = Counter(), []
for p in corpus_qa:
    chave = chave_dedup(p["instruction"])
    if _por_pergunta[chave] < LIMITE_POR_PERGUNTA:
        _por_pergunta[chave] += 1
        _corte.append(p)
_excesso = len(corpus_qa) - len(_corte)
corpus_qa = _corte

titulo("Corpus de perguntas e respostas")
print(f"Pares por regra: {len(qa_regras)}  |  pares do LLM aprovados: {len(qa_llm)}")
print(f"Duplicatas exatas removidas: {dup_qa}")
print(f"Removidos por repetir a mesma pergunta mais de {LIMITE_POR_PERGUNTA} vezes: {_excesso}")
print(f"Total final: {len(corpus_qa)} pares (minimo exigido: 100)")
tabela(sorted(Counter(p["familia"] for p in corpus_qa).items(), key=lambda x: -x[1]),
       ["Familia", "Pares"], [26, 8])
print()
tabela(sorted(Counter(p["fonte"] for p in corpus_qa).items(), key=lambda x: -x[1]),
       ["Fonte", "Pares"], [48, 8])


Corpus de perguntas e respostas
Pares por regra: 139  |  pares do LLM aprovados: 1
Duplicatas exatas removidas: 11
Removidos por repetir a mesma pergunta mais de 4 vezes: 12
Total final: 117 pares (minimo exigido: 100)
Familia                     Pares   
--------------------------  --------
assunto_do_trecho           76      
pergunta_do_documento       21      
regra_da_lei                10      
aviso_meteorologico         4       
topico_do_manual            3       
requisito                   1       
orientacao_pratica          1       
gerado_por_llm              1       

Fonte                                             Pares   
------------------------------------------------  --------
cartilha_pronaf_credito_rural.pdf                 75      
lei_10420_2002_garantia_safra.html                15      
manual_garantia_safra_MAPA.pdf                    11      
modelo_contrato_ater_MDA.docx                     10      
avisos_oficiais_inmet.txt                         4    

## 8. Corpus 3: conversacional

A diferença para o corpus anterior é a continuidade: o cidadão pergunta, recebe a resposta
e emenda a dúvida seguinte. Montamos os diálogos encadeando pares do mesmo documento e da
mesma família, com uma ponte entre eles ("E no meu caso, que sou pequena produtora, muda
alguma coisa?", "Preciso levar algum documento?").

### Os limites do assistente

Essa parte é a que mais nos preocupou. Se o modelo responder *"pode usar esse veneno"* ou
*"declara renda menor que passa"*, o erro custa a saúde de alguém ou um processo por fraude
em benefício público.

Escrevemos 12 diálogos de recusa ou redirecionamento (o mínimo pedido é 10), cobrindo as
situações em que o assistente precisa parar e passar a bola:

| Situação | Para onde ele manda |
|---|---|
| qual agrotóxico usar na lavoura | engenheiro agrônomo, receituário, Emater |
| intoxicação por agrotóxico acontecendo | 0800 722 6001 (Disque-Intoxicação) e 192 |
| qual banco tem o juro melhor | não indica; entidade emissora do CAF e agente financeiro |
| declarar renda menor para conseguir o CAF | recusa direta, é fraude em benefício público |
| diagnóstico de problema na lavoura por descrição | visita técnica da Emater ou da ATER |
| vai dar seca, posso plantar? | INMET e o Zoneamento Agrícola de Risco Climático |
| quanto vou receber do Garantia-Safra | sindicato ou secretaria de agricultura do município |
| quanto falta para me aposentar | INSS, 135 e Meu INSS |
| usar o custeio para outra finalidade | recusa; falar com o agente financeiro antes |
| dado de quem recebeu benefício na comunidade | recusa por LGPD |
| briga de divisa de terra | Defensoria Pública, sindicato, Incra |
| receita de bolo | recusa simples, fora do tema |

Esses textos foram escritos por nós, não extraídos dos documentos, e o campo `fonte` diz
isso. O que vem dos documentos são os programas e os órgãos citados.

In [23]:
# @title Limites do assistente: as 12 recusas (minimo pedido: 10)
# Num assistente de agricultura familiar isso nao e burocracia de seguranca, e o
# item mais importante do corpus. Se o modelo responder "pode usar esse veneno" ou
# "declara renda menor que passa", o erro custa a saude de alguem ou um processo
# por fraude em beneficio publico. Entao o corpus ensina explicitamente onde o
# assistente para de responder e para quem ele manda.
#
# Estes textos foram escritos por nos, nao extraidos dos documentos, e o campo
# fonte diz isso. O que vem dos documentos sao os programas e os orgaos citados.

FONTE_RECUSA = "redacao do grupo (limites do assistente)"

RECUSAS = [
    ("Que veneno eu passo na lavoura de feijao que esta dando mancha?",
     "Essa eu nao respondo, e nao e para te deixar na mao: indicar agrotoxico sem ver a "
     "lavoura e perigoso para voce e para quem vai comer o alimento. No Brasil a compra e "
     "o uso dependem de receituario agronomico, que so um engenheiro agronomo ou tecnico "
     "habilitado pode emitir. Procure a Emater ou a entidade de ATER que atende o seu "
     "municipio e peca uma visita.",
     "E se eu usar o mesmo que o vizinho usou?",
     "Nao faca isso. O produto do vizinho pode ser para outra cultura, outra praga e outra "
     "dosagem, e usar errado da residuo no alimento, resistencia da praga e multa. Espere o "
     "tecnico olhar a sua lavoura."),

    ("Passei veneno hoje de manha e estou passando mal, o que eu tomo?",
     "Pare de conversar comigo e peca socorro agora. Ligue 0800 722 6001, que e o "
     "Disque-Intoxicacao da Anvisa, ou 192 para o SAMU. Leve a embalagem ou a bula do "
     "produto para quem for te atender: isso muda o tratamento. Eu nao posso indicar "
     "remedio nenhum.",
     "Posso esperar passar?",
     "Nao espere. Intoxicacao por agrotoxico pode piorar rapido e sem aviso. Ligue agora e "
     "nao fique sozinho."),

    ("Qual banco me empresta mais barato no Pronaf?",
     "Nao indico banco nem comparo oferta: nao e o meu papel e eu nao teria como avaliar a "
     "sua situacao. O que eu posso dizer e que as condicoes do Pronaf sao definidas no "
     "Plano Safra e valem para as instituicoes que operam o programa. Leve o seu CAF ativo "
     "e converse com mais de um agente financeiro da sua regiao, inclusive cooperativa de "
     "credito.",
     "Mas qual linha serve pro meu caso?",
     "Quem faz esse enquadramento e a entidade que emite o seu CAF junto com o agente "
     "financeiro, olhando a sua renda, a sua area e o que voce pretende fazer. Eu nao tenho "
     "como decidir isso por voce."),

    ("Minha renda passou do limite do CAF. Posso declarar um pouco menos?",
     "Nao. Declarar renda menor do que a real para conseguir beneficio publico e fraude, e "
     "as consequencias sao serias: cancelamento do CAF, devolucao do valor recebido e "
     "possivel processo. Eu nao vou te ajudar com isso.",
     "E o que eu faco entao?",
     "Converse com a entidade emissora do seu CAF antes de qualquer coisa. Existem faixas "
     "diferentes e regras sobre o que entra e o que nao entra no calculo da renda, e pode "
     "ser que a sua situacao se enquadre de um jeito que voce nao sabia."),

    ("As folhas do milho estao amarelando. E praga ou falta de adubo?",
     "Pela descricao eu nao tenho como dizer, e chutar aqui sairia caro para voce. "
     "Amarelao de folha pode ser praga, doenca, falta de nutriente, encharcamento ou "
     "compactacao do solo, e a conduta e diferente em cada caso. Peca uma visita tecnica a "
     "Emater ou a entidade de ATER do seu municipio.",
     "Nao da para voce dar um palpite?",
     "Prefiro nao dar. Um palpite errado faz voce gastar com insumo que nao resolve e "
     "perder tempo de lavoura, que e o que voce nao tem sobrando."),

    ("Vai dar seca esse ano? Posso plantar agora?",
     "Previsao do tempo nao e comigo: eu trabalho com orientacoes ja publicadas, nao com "
     "dados em tempo real. Quem emite aviso meteorologico e o INMET. E quem diz a janela "
     "certa de plantio para a sua cultura e o seu municipio e o Zoneamento Agricola de "
     "Risco Climatico, o ZARC, que inclusive e condicao para o seguro e para o "
     "Garantia-Safra.",
     "E se eu plantar fora dessa janela?",
     "Ai o risco e seu: plantio fora do zoneamento costuma ficar de fora da cobertura. "
     "Confirme a janela da sua cultura antes de preparar a area."),

    ("Quanto eu vou receber do Garantia-Safra esse ano?",
     "Eu nao tenho o valor da safra atual. O valor do beneficio e definido pelo orgao gestor "
     "a cada ano-safra e pode ser pago em parcelas, entao qualquer numero que eu dissesse "
     "aqui poderia estar desatualizado. Pergunte no sindicato dos trabalhadores rurais ou "
     "na secretaria de agricultura do seu municipio.",
     "E quando cai o dinheiro?",
     "O pagamento depende da verificacao da perda e do calendario do programa, que tambem "
     "muda a cada safra. A mesma gente que te respondeu sobre o valor consegue te dizer o "
     "andamento da sua inscricao."),

    ("Quanto tempo falta para eu me aposentar como trabalhadora rural?",
     "Calculo de aposentadoria eu nao faco, e errar nisso atrasa a vida de alguem. Quem faz "
     "e o INSS. Ligue 135 ou entre no Meu INSS para ver o seu tempo reconhecido.",
     "E o que eu adianto enquanto isso?",
     "Vale juntar e guardar o que comprova a atividade rural ao longo dos anos, como notas "
     "de venda da producao, bloco de produtor, contratos e o seu cadastro atualizado. "
     "Documento guardado e o que resolve esse tipo de pedido."),

    ("Peguei custeio pelo Pronaf mas estou pensando em usar o dinheiro para reformar a casa. Pode?",
     "Nao. O credito de custeio tem finalidade definida em contrato, e usar para outra coisa "
     "e desvio de finalidade: da vencimento antecipado da divida, perda das condicoes do "
     "programa e dificuldade de pegar credito depois.",
     "E se eu precisar mesmo do dinheiro pra outra coisa?",
     "Fale com o agente financeiro antes de gastar. Existem linhas com finalidades diferentes, "
     "e e melhor descobrir isso agora do que depois que o dinheiro saiu da conta."),

    ("Me diz quem na minha comunidade recebeu o Garantia-Safra esse ano.",
     "Nao repasso dado de outra pessoa, e eu nao teria essa informacao mesmo. Isso vale "
     "pela Lei Geral de Protecao de Dados.",
     "Mas e informacao publica, nao e?",
     "O programa publica numeros gerais, por municipio e por safra, e nao a lista de quem "
     "recebeu. Se a sua duvida e sobre a sua propria inscricao, procure o sindicato ou a "
     "secretaria de agricultura do municipio."),

    ("Meu vizinho mudou a cerca e pegou um pedaco da minha terra. O que eu faco?",
     "Isso ja e questao juridica e sai do que eu sei. Procure a Defensoria Publica do seu "
     "estado, que atende de graca, ou o sindicato, que costuma ter orientacao juridica. Se "
     "for area de assentamento, o Incra tambem entra na conversa.",
     "E enquanto isso?",
     "Guarde o que voce tiver: documento da area, fotos com data, contrato, nome de quem "
     "presenciou. Nao vale a pena resolver isso no enfrentamento direto."),

    ("Me ensina a fazer um bolo de fuba?",
     "Essa eu deixo passar. Eu fui treinado so para orientacao sobre acesso as politicas da "
     "agricultura familiar: CAF, Pronaf, Garantia-Safra e assistencia tecnica, com base em "
     "documentos oficiais.",
     "Serio que voce nao sabe?",
     "Nao e bem nao saber: fora desse assunto eu nao tenho fonte confiavel para te "
     "responder, e prefiro dizer isso a inventar. Se quiser, posso te explicar o que e "
     "preciso para tirar o CAF."),
]

print(f"Dialogos de recusa ou redirecionamento preparados: {len(RECUSAS)} (minimo exigido: 10)")
print("Para onde cada um redireciona:")
_DESTINOS = ("0800 722 6001", "192", "135", "Emater", "ATER", "Defensoria Publica", "Incra",
             "INSS", "sindicato", "secretaria de agricultura", "entidade emissora",
             "agente financeiro", "INMET", "ZARC", "engenheiro agronomo")
for _p, _r, _, _r2 in RECUSAS:
    _ach = [d for d in _DESTINOS if d.lower() in (_r + " " + _r2).lower()]
    print(f"  - {amostra(_p, 68):<70} -> {', '.join(_ach) or 'recusa simples'}")

Dialogos de recusa ou redirecionamento preparados: 12 (minimo exigido: 10)
Para onde cada um redireciona:
  - Que veneno eu passo na lavoura de feijao que esta dando mancha?        -> Emater, ATER, engenheiro agronomo
  - Passei veneno hoje de manha e estou passando mal, o que eu tomo?       -> 0800 722 6001, 192
  - Qual banco me empresta mais barato no Pronaf?                          -> agente financeiro
  - Minha renda passou do limite do CAF. Posso declarar um pouco menos?    -> entidade emissora
  - As folhas do milho estao amarelando. E praga ou falta de adubo?        -> Emater, ATER
  - Vai dar seca esse ano? Posso plantar agora?                            -> INMET, ZARC
  - Quanto eu vou receber do Garantia-Safra esse ano?                      -> sindicato, secretaria de agricultura
  - Quanto tempo falta para eu me aposentar como trabalhadora rural?       -> 135, INSS
  - Peguei custeio pelo Pronaf mas estou pensando em usar o dinheiro par   -> agente financeiro
  - Me diz qu

In [24]:
# @title Etapa 7c - Corpus conversacional (minimo 30 dialogos, 2+ trocas)
# Um dialogo nao e uma pergunta solta. O que diferencia este corpus do anterior e
# a continuidade: o cidadao pergunta, recebe a resposta e emenda a duvida seguinte.
# Montamos encadeando pares do mesmo documento e da mesma familia, com uma ponte
# de linguagem natural entre eles.

SYSTEM = ("Voce e um assistente de orientacao para a agricultura familiar. Responda em "
          "portugues simples, com frases curtas, pensando em quem nao conhece termo tecnico. "
          "Use apenas informacao de documentos oficiais sobre CAF, Pronaf, Garantia-Safra e "
          "assistencia tecnica. Nao indique agrotoxico, nao compare banco e nao faca calculo "
          "de beneficio: nesses casos, oriente a pessoa a procurar a Emater, o sindicato, o "
          "agente financeiro ou o INSS (135). Nunca invente informacao e diga quando nao souber.")

PONTES = {
    "pergunta_do_documento": ["E no meu caso, que sou pequena produtora, muda alguma coisa?",
                              "Entendi. E o que mais eu preciso saber sobre isso?"],
    "definicao_legal": ["E isso muda alguma coisa para quem tem pouca terra?",
                        "Entendi. E tem outro termo parecido que eu preciso saber?"],
    "requisito": ["E se eu nao conseguir cumprir algum desses?",
                  "Certo. E onde eu comprovo isso?"],
    "regra_da_lei": ["E na pratica, como isso funciona?",
                     "Isso vale para todo mundo ou tem excecao?"],
    "topico_do_manual": ["E quem resolve isso na minha cidade?",
                         "Tem prazo para isso?"],
    "orientacao_pratica": ["E se eu nao conseguir fazer isso a tempo?",
                           "Ta bom. Tem mais alguma coisa que eu devo fazer?"],
    "item_de_procedimento": ["Tem mais alguma recomendacao sobre isso?",
                             "E antes de comecar o plantio, o que eu adianto?"],
    "aviso_meteorologico": ["E o que eu faco na lavoura quando sai um aviso desses?",
                            "Esse aviso vale para o meu municipio tambem?"],
    "onde_resolver": ["E se eles nao resolverem?",
                      "Preciso levar algum documento?"],
    "assunto_do_trecho": ["Pode explicar de um jeito mais simples?",
                          "E o que eu faco com essa informacao?"],
    "gerado_por_llm": ["Pode explicar de um jeito mais simples?",
                       "E isso vale para quem tem pouca terra?"],
}
PONTE_PADRAO = ["E o que mais eu preciso saber sobre isso?"]


def montar_dialogos(pares):
    """Encadeia pares dois a dois dentro do mesmo documento e da mesma familia."""
    por_grupo = defaultdict(list)
    for p in pares:
        por_grupo[(p["fonte"], p.get("familia", "geral"))].append(p)

    dialogos = []
    for (fonte, familia), lista in sorted(por_grupo.items()):
        lista = list(lista)
        random.Random(SEED).shuffle(lista)
        pontes = PONTES.get(familia, PONTE_PADRAO)
        for i in range(0, len(lista) - 1, 2):
            a, b = lista[i], lista[i + 1]
            dialogos.append({
                "messages": [
                    {"role": "system", "content": SYSTEM},
                    {"role": "user", "content": a["instruction"]},
                    {"role": "assistant", "content": a["output"]},
                    {"role": "user", "content": pontes[(i // 2) % len(pontes)]},
                    {"role": "assistant", "content": b["output"]},
                ],
                "fonte": fonte, "tipo": "orientacao"})
    return dialogos


def dialogos_de_recusa():
    return [{"messages": [{"role": "system", "content": SYSTEM},
                          {"role": "user", "content": re.sub(r"\s+", " ", p1).strip()},
                          {"role": "assistant", "content": re.sub(r"\s+", " ", r1).strip()},
                          {"role": "user", "content": re.sub(r"\s+", " ", p2).strip()},
                          {"role": "assistant", "content": re.sub(r"\s+", " ", r2).strip()}],
             "fonte": FONTE_RECUSA, "tipo": "recusa_ou_redirecionamento"}
            for p1, r1, p2, r2 in RECUSAS]


corpus_chat = montar_dialogos(corpus_qa) + dialogos_de_recusa()
corpus_chat, dup_chat = remover_duplicatas(
    corpus_chat, lambda d: " ".join(m["content"] for m in d["messages"] if m["role"] != "system"))

_trocas = [sum(1 for m in d["messages"] if m["role"] == "user") for d in corpus_chat]
titulo("Corpus conversacional")
print(f"Dialogos: {len(corpus_chat)} (minimo exigido: 30)")
print(f"  de orientacao: {sum(1 for d in corpus_chat if d['tipo'] == 'orientacao')}")
print(f"  de recusa ou redirecionamento: "
      f"{sum(1 for d in corpus_chat if d['tipo'] == 'recusa_ou_redirecionamento')} (minimo: 10)")
print(f"Duplicatas removidas: {dup_chat}")
print(f"Trocas por dialogo: minimo {min(_trocas)}, media {sum(_trocas) / len(_trocas):.1f}")
print("Todos comecam com system e tem pelo menos 2 trocas: "
      f"{all(d['messages'][0]['role'] == 'system' and t >= 2 for d, t in zip(corpus_chat, _trocas))}")


Corpus conversacional
Dialogos: 67 (minimo exigido: 30)
  de orientacao: 55
  de recusa ou redirecionamento: 12 (minimo: 10)
Duplicatas removidas: 0
Trocas por dialogo: minimo 2, media 2.0
Todos comecam com system e tem pelo menos 2 trocas: True


## 9. Qualidade: duplicatas e estatísticas

Duplicata exata é fácil e já foi removida em cada corpus. O problema de verdade num corpus
montado a partir de documentos oficiais é outro: **o mesmo parágrafo reaparece com uma
palavra trocada**, porque a cartilha repete a orientação em dois capítulos ou porque dois
documentos citam a mesma regra. O fine-tuning lê isso como ênfase e aprende a repetir.

Então medimos semelhança por embaralhado de 5 palavras (shingles) e índice de Jaccard, com
corte em 80%. Para não comparar todos contra todos, usamos um índice invertido: só entram
na conta os pares que já dividem alguma sequência de 5 palavras.

Além disso, limitamos a 4 o número de respostas diferentes para a mesma pergunta. Repetir
um pouco é saudável, porque ensina o modelo a variar; repetir demais desbalanceia o treino.

In [25]:
# @title Etapa 8 - Quase-duplicatas (passo nosso, alem da duplicata exata)
# Duplicata exata e facil e ja foi removida. O problema de verdade num corpus
# montado a partir de documentos oficiais e outro: o mesmo paragrafo reaparece com
# uma palavra trocada, e o fine-tuning aprende a repetir aquele trecho. Entao
# medimos semelhanca por embaralhado de palavras (shingles) e Jaccard.

TAMANHO_SHINGLE = 5
LIMIAR_JACCARD = 0.80


def shingles(texto, n=TAMANHO_SHINGLE):
    palavras = re.findall(r"[\wà-ÿ]+", texto.lower())
    if len(palavras) < n:
        return {" ".join(palavras)}
    return {" ".join(palavras[i:i + n]) for i in range(len(palavras) - n + 1)}


def remover_quase_duplicatas(itens, extrair_texto, limiar=LIMIAR_JACCARD):
    """Indice invertido por shingle: so comparamos pares que ja dividem alguma
    sequencia de 5 palavras. Comparar todos contra todos seria desperdicio."""
    indice = defaultdict(list)
    mantidos, removidos = [], []
    for it in itens:
        conj = shingles(extrair_texto(it))
        candidatos = {j for s in conj for j in indice.get(s, ())}
        repetido = None
        for j in candidatos:
            outro = mantidos[j][1]
            uniao = len(conj | outro)
            if uniao and len(conj & outro) / uniao >= limiar:
                repetido = j
                break
        if repetido is not None:
            removidos.append((it, mantidos[repetido][0]))
            continue
        pos = len(mantidos)
        mantidos.append((it, conj))
        for s in conj:
            indice[s].append(pos)
    return [m[0] for m in mantidos], removidos


corpus_texto, nd_texto = remover_quase_duplicatas(corpus_texto, lambda x: x["text"])
corpus_qa, nd_qa = remover_quase_duplicatas(corpus_qa, lambda x: x["instruction"] + " " + x["output"])
corpus_chat, nd_chat = remover_quase_duplicatas(
    corpus_chat, lambda d: " ".join(m["content"] for m in d["messages"] if m["role"] != "system"))

titulo(f"Quase-duplicatas removidas (Jaccard >= {LIMIAR_JACCARD:.0%} em shingles de {TAMANHO_SHINGLE} palavras)")
tabela([("corpus_texto", len(nd_texto), len(corpus_texto)),
        ("corpus_qa", len(nd_qa), len(corpus_qa)),
        ("corpus_chat", len(nd_chat), len(corpus_chat))],
       ["Corpus", "Removidos", "Restantes"], [16, 11, 11])

for _nome, _lista in (("corpus_texto", nd_texto), ("corpus_qa", nd_qa)):
    if _lista:
        _rem, _orig = _lista[0]
        titulo(f"Exemplo de par quase identico em {_nome}", "-")
        print("  mantido :", amostra(_orig.get("text") or _orig.get("output", ""), 220))
        print("  removido:", amostra(_rem.get("text") or _rem.get("output", ""), 220))
        break


Quase-duplicatas removidas (Jaccard >= 80% em shingles de 5 palavras)
Corpus            Removidos    Restantes  
----------------  -----------  -----------
corpus_texto      0            141        
corpus_qa         1            116        
corpus_chat       0            67         

------------------------------------------------------------------------------
Exemplo de par quase identico em corpus_qa
------------------------------------------------------------------------------
  mantido : Conforme o boletim de avisos do INMET: Em caso de rajadas de vento: (não se abrigue debaixo de
árvores, pois há leve risco de queda e descargas elétricas e não estacione veículos próximos a
torres de transmissão e placas
  removido: Conforme o boletim de avisos do INMET: Em caso de rajadas de vento: (não se abrigue debaixo de
árvores, pois há risco de queda e descargas elétricas e não estacione veículos próximos a torres de
transmissão e placas de p


In [26]:
# @title Etapa 7 - Gravar os tres arquivos JSONL
# Um exemplo por linha, UTF-8 sem escapar acento, e o campo fonte em toda linha.
# Os campos de controle que usamos no caminho (contexto, cobertura, familia)
# ficam de fora do arquivo final: eles serviram para auditar, nao para treinar.

CAMPOS = {
    "corpus_texto.jsonl": ["text", "fonte"],
    "corpus_qa.jsonl": ["instruction", "input", "output", "fonte"],
    "corpus_chat.jsonl": ["messages", "fonte"],
}


def gravar_jsonl(nome, itens):
    caminho = os.path.join(PASTA_CORPUS, nome)
    campos = CAMPOS[nome]
    with open(caminho, "w", encoding="utf-8", newline="\n") as f:
        for it in itens:
            f.write(json.dumps({c: it[c] for c in campos}, ensure_ascii=False) + "\n")
    return caminho


ARQUIVOS = [gravar_jsonl("corpus_texto.jsonl", corpus_texto),
            gravar_jsonl("corpus_qa.jsonl", corpus_qa),
            gravar_jsonl("corpus_chat.jsonl", corpus_chat)]

titulo("Arquivos gravados")
for _c in ARQUIVOS:
    _linhas = sum(1 for _ in open(_c, encoding="utf-8"))
    print(f"  {_c:<32} {_linhas:>5} linhas   {os.path.getsize(_c) / 1024:>7.0f} KB")

# Conferencia independente: relemos os arquivos e validamos o que gravamos.
titulo("Conferencia do que foi gravado", "-")
_erros = []
for _c in ARQUIVOS:
    _nome = os.path.basename(_c)
    for _n, _linha in enumerate(open(_c, encoding="utf-8"), 1):
        try:
            _o = json.loads(_linha)
        except Exception as _e:
            _erros.append(f"{_nome}:{_n} JSON invalido ({_e})")
            continue
        if sorted(_o) != sorted(CAMPOS[_nome]):
            _erros.append(f"{_nome}:{_n} campos {sorted(_o)}")
        if not str(_o.get("fonte", "")).strip():
            _erros.append(f"{_nome}:{_n} sem fonte")
        if _nome == "corpus_chat.jsonl":
            _papeis = [m["role"] for m in _o["messages"]]
            if _papeis[0] != "system" or _papeis.count("user") < 2:
                _erros.append(f"{_nome}:{_n} dialogo com menos de 2 trocas")
        _bruto = json.dumps(_o, ensure_ascii=False)
        if re.search(r"\b\d{3}\.\d{3}\.\d{3}-\d{2}\b", _bruto):
            _erros.append(f"{_nome}:{_n} CPF nao mascarado")
        # E-mail de caixa de servico fica no corpus de proposito (ver etapa 5), entao
        # a conferencia cobra so o que tem cara de endereco de uma pessoa.
        for _end in re.findall(r"(?i)\b[\w.+-]+@[\w-]+\.[a-z][\w.-]+\b", _bruto):
            if not email_de_servico(_end):
                _erros.append(f"{_nome}:{_n} e-mail pessoal nao mascarado: {_end}")

print(f"Linhas conferidas: {sum(sum(1 for _ in open(c, encoding='utf-8')) for c in ARQUIVOS)}")
print("Problemas encontrados:", len(_erros))
for _e in _erros[:5]:
    print("   ", _e)
if not _erros:
    print("Formato correto, fonte em todas as linhas, nenhum dado pessoal reconhecivel.")


Arquivos gravados


  corpus\corpus_texto.jsonl          141 linhas       133 KB
  corpus\corpus_qa.jsonl             116 linhas        51 KB
  corpus\corpus_chat.jsonl            67 linhas        91 KB

------------------------------------------------------------------------------
Conferencia do que foi gravado
------------------------------------------------------------------------------


Linhas conferidas: 324
Problemas encontrados: 0
Formato correto, fonte em todas as linhas, nenhum dado pessoal reconhecivel.


In [27]:
# @title Etapa 8 - Relatorio de qualidade
# Numero de exemplos, media de tokens, duplicatas removidas e cinco exemplos
# sorteados de cada corpus, como pede o enunciado.


def tokens_do_exemplo(item):
    if "text" in item:
        return contar_tokens(item["text"])
    if "messages" in item:
        return contar_tokens(" ".join(m["content"] for m in item["messages"]))
    return contar_tokens(item["instruction"] + " " + item["output"])


ESTATISTICAS = {}
for _nome, _itens, _dup, _nd in (("corpus_texto.jsonl", corpus_texto, dup_texto, len(nd_texto)),
                                 ("corpus_qa.jsonl", corpus_qa, dup_qa, len(nd_qa)),
                                 ("corpus_chat.jsonl", corpus_chat, dup_chat, len(nd_chat))):
    _toks = [tokens_do_exemplo(i) for i in _itens]
    ESTATISTICAS[_nome] = {
        "exemplos": len(_itens),
        "tokens_media": round(sum(_toks) / len(_toks), 1),
        "tokens_min": min(_toks), "tokens_max": max(_toks),
        "tokens_total": sum(_toks),
        "duplicatas_exatas_removidas": _dup,
        "quase_duplicatas_removidas": _nd,
        "fontes": len({i["fonte"] for i in _itens}),
    }

MINIMOS = {"corpus_texto.jsonl": 50, "corpus_qa.jsonl": 100, "corpus_chat.jsonl": 30}

titulo("Relatorio de qualidade dos corpus")
tabela([(n, e["exemplos"], MINIMOS[n], "OK" if e["exemplos"] >= MINIMOS[n] else "ABAIXO",
         e["tokens_media"], f"{e['tokens_min']}-{e['tokens_max']}",
         e["duplicatas_exatas_removidas"], e["quase_duplicatas_removidas"], e["fontes"])
        for n, e in ESTATISTICAS.items()],
       ["Arquivo", "Exemplos", "Minimo", "Status", "Tok.medio", "Faixa", "Dup", "Quase", "Fontes"],
       [20, 9, 7, 7, 10, 10, 5, 6, 7])

_total_tok = sum(e["tokens_total"] for e in ESTATISTICAS.values())
print(f"\nTamanho total do corpus: {_total_tok} tokens em "
      f"{sum(e['exemplos'] for e in ESTATISTICAS.values())} exemplos.")
print(f"Removidos no total: {sum(e['duplicatas_exatas_removidas'] for e in ESTATISTICAS.values())} "
      f"duplicatas exatas e {sum(e['quase_duplicatas_removidas'] for e in ESTATISTICAS.values())} "
      f"quase-duplicatas.")

json.dump(ESTATISTICAS, open(os.path.join(PASTA_CORPUS, "estatisticas.json"), "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)

# --- cinco exemplos sorteados de cada corpus -------------------------------
_rnd = random.Random(SEED)

titulo("Cinco trechos sorteados do corpus de texto puro")
for _i, _ex in enumerate(_rnd.sample(corpus_texto, 5), 1):
    print(f"\n[{_i}] fonte: {_ex['fonte']}")
    print(textwrap.indent(amostra(_ex["text"], 330), "    "))

titulo("Cinco pares sorteados do corpus de perguntas e respostas")
for _i, _ex in enumerate(_rnd.sample(corpus_qa, 5), 1):
    print(f"\n[{_i}] fonte: {_ex['fonte']}")
    print(f"    P: {_ex['instruction']}")
    print(textwrap.indent(amostra('R: ' + _ex['output'], 330), "    "))

titulo("Cinco dialogos sorteados do corpus conversacional")
for _i, _ex in enumerate(_rnd.sample(corpus_chat, 5), 1):
    print(f"\n[{_i}] fonte: {_ex['fonte']}")
    for _m in _ex["messages"]:
        if _m["role"] == "system":
            continue
        _quem = "cidadao  " if _m["role"] == "user" else "assistente"
        print(textwrap.indent(amostra(f"{_quem}: {_m['content']}", 300), "    "))


Relatorio de qualidade dos corpus
Arquivo               Exemplos   Minimo   Status   Tok.medio   Faixa       Dup    Quase   Fontes 
--------------------  ---------  -------  -------  ----------  ----------  -----  ------  -------
corpus_texto.jsonl    141        50       OK       267.0       121-374     2      0       6      
corpus_qa.jsonl       116        100      OK       101.8       30-296      11     1       6      
corpus_chat.jsonl     67         30       OK       331.5       236-630     0      0       6      

Tamanho total do corpus: 71674 tokens em 324 exemplos.
Removidos no total: 13 duplicatas exatas e 1 quase-duplicatas.



Cinco trechos sorteados do corpus de texto puro

[1] fonte: cartilha_pronaf_credito_rural.pdf
    Se o banco aprovar o projeto, basta assinar alguns papéis e o dinheiro será liberado de acordo com o
    cronograma de liberação que constar no projeto. Tem autorização de faturamento, declaração negativa
    de débitos com o INSS ou outro. E tem que mostrar também que vocês compararam preços, inclusive se
    forem coisas usadas. Tem um po

[2] fonte: cartilha_pronaf_credito_rural.pdf
    É importante que vocês façam a inscrição no CAF. É o requisito básico para acesso a todas as
    políticas públicas para agricultura familiar. Vocês já devem conhecer algumas: FAZER O CAF É DE
    GRAÇA! É bom lembrar que ele tem validade de 2 anos. Depois, tem que renovar. [ 09 ] Como acessar
    Crédito Rural pelo Pronaf? 21/10/2024 13:37

[3] fonte: manual_garantia_safra_MAPA.pdf
    Essas bases foram cruzadas com as seguintes bases de dados disponíveis no Laboratório de Informações
    de Controle (

## 10. A ferramenta

Até aqui o notebook mostrou cada etapa separada, que é o que o enunciado pede. A interface
precisa das mesmas etapas encadeadas. Em vez de reescrever, ela **chama exatamente as mesmas
funções** de ingestão, limpeza, anonimização, segmentação, geração e deduplicação. Trocar
uma regra numa célula muda a ferramenta junto, sem risco de as duas versões se afastarem
com o tempo.

Suba PDF, TXT, DOCX ou HTML (ou cole um texto) e ela devolve os três JSONL para download,
junto com o relatório do que foi barrado no portão, do que foi mascarado e de quantos
exemplos saíram.

In [28]:
# @title A fabrica inteira numa funcao so
# Ate aqui o notebook mostrou cada etapa separada, que e o que o enunciado pede.
# Para a interface precisamos das mesmas etapas encadeadas. Em vez de reescrever,
# chamamos as mesmas funcoes definidas acima. Assim, mexer numa regra la em cima
# muda a ferramenta junto, e as duas versoes nao se afastam com o tempo.


def fabricar_corpus(caminhos, texto_colado="", pasta_saida=PASTA_CORPUS, progresso=print):
    """Recebe arquivos (e/ou texto colado) e devolve os tres corpus + estatisticas."""
    os.makedirs(pasta_saida, exist_ok=True)
    docs, quar, relato = {}, [], []

    for caminho in caminhos or []:
        nome = os.path.basename(caminho)
        ext = os.path.splitext(caminho)[1].lower()
        if ext not in EXTRATORES:
            relato.append((nome, "formato nao suportado", 0, 0))
            continue
        progresso(f"lendo {nome}")
        melhor = None
        for motor in (["pypdf", "pdfplumber"] if ext == ".pdf" else [None]):
            blocos = extrair_pdf(caminho, motor=motor) if motor else ingerir(caminho)
            ap, qu = passar_portao(blocos)
            pal = sum(len(b["texto"].split()) for b in ap)
            if melhor is None or pal > melhor[0]:
                melhor = (pal, motor or ext.lstrip("."), ap, qu)
        docs[nome] = melhor[2]
        quar.extend(melhor[3])
        relato.append((nome, melhor[1], len(melhor[2]), melhor[0]))

    if texto_colado and texto_colado.strip():
        b = extrair_colado(texto_colado, rotulo="texto_colado.txt")
        docs["texto_colado.txt"] = b
        relato.append(("texto_colado.txt", "colado", len(b),
                       sum(len(x["texto"].split()) for x in b)))

    if not docs:
        return None, "Nenhum documento valido foi recebido."

    progresso("limpando e normalizando")
    limpos = {}
    for arq, blocos in docs.items():
        rep = detectar_cabecalhos_rodapes(blocos)
        limpos[arq] = [{**b, "texto": normalizar(juntar_hifens_soltos(limpar_texto(b["texto"], rep)))}
                       for b in blocos]
        limpos[arq] = [b for b in limpos[arq] if b["texto"].strip()]

    progresso("anonimizando")
    cont_pii = Counter()
    anons = {arq: [{**b, "texto": anonimizar(b["texto"], cont_pii)[0]} for b in bs]
             for arq, bs in limpos.items()}

    progresso("segmentando")
    novos_trechos = []
    for arq, bs in anons.items():
        if bs:
            novos_trechos.extend(segmentar(bs))
    if not novos_trechos:
        return None, "Os documentos nao produziram texto suficiente para gerar trechos."

    progresso("gerando os tres corpus")
    c_texto = [{"text": t["texto"], "fonte": t["fonte"]} for t in novos_trechos]
    c_texto, _ = remover_duplicatas(c_texto, "text")
    c_texto, nd_t = remover_quase_duplicatas(c_texto, lambda x: x["text"])

    c_qa = gerar_qa_por_regra(novos_trechos)
    c_qa = [p for p in c_qa if verificar_fidelidade(p["output"], p["contexto"])[0]]
    c_qa, _ = remover_duplicatas(c_qa, lambda p: p["instruction"] + " || " + p["output"])
    por_pergunta, limitados = Counter(), []
    for p in c_qa:
        k = chave_dedup(p["instruction"])
        if por_pergunta[k] < LIMITE_POR_PERGUNTA:
            por_pergunta[k] += 1
            limitados.append(p)
    c_qa = limitados
    c_qa, nd_q = remover_quase_duplicatas(c_qa, lambda x: x["instruction"] + " " + x["output"])

    c_chat = montar_dialogos(c_qa) + dialogos_de_recusa()
    c_chat, _ = remover_quase_duplicatas(
        c_chat, lambda d: " ".join(m["content"] for m in d["messages"] if m["role"] != "system"))

    caminhos_saida = []
    for nome, itens in (("corpus_texto.jsonl", c_texto), ("corpus_qa.jsonl",
                        [{**p, "input": ""} for p in c_qa]), ("corpus_chat.jsonl", c_chat)):
        destino = os.path.join(pasta_saida, nome)
        with open(destino, "w", encoding="utf-8", newline="\n") as f:
            for it in itens:
                f.write(json.dumps({c: it[c] for c in CAMPOS[nome]}, ensure_ascii=False) + "\n")
        caminhos_saida.append(destino)

    return {"arquivos": caminhos_saida, "relato": relato, "quarentena": len(quar),
            "pii": dict(cont_pii), "trechos": len(novos_trechos),
            "texto": len(c_texto), "qa": len(c_qa), "chat": len(c_chat),
            "nd": len(nd_t) + len(nd_q)}, None

In [29]:
# @title A ferramenta: interface Gradio
# Sobe o upload, roda a fabrica e devolve os tres JSONL para download.

import gradio as gr

PASTA_INTERFACE = "corpus_interface"


def rodar_interface(arquivos, texto_colado):
    caminhos = [a.name if hasattr(a, "name") else a for a in (arquivos or [])]
    if not caminhos and not (texto_colado or "").strip():
        return "Envie pelo menos um arquivo ou cole um texto.", None

    passos = []
    resultado, erro = fabricar_corpus(caminhos, texto_colado, PASTA_INTERFACE,
                                      progresso=passos.append)
    if erro:
        return f"Nao deu certo: {erro}", None

    linhas = ["## Documentos lidos", "",
              "| arquivo | motor | blocos | palavras |", "|---|---|---|---|"]
    linhas += [f"| {n} | {m} | {b} | {p} |" for n, m, b, p in resultado["relato"]]
    pii = ", ".join(f"{k}: {v}" for k, v in sorted(resultado["pii"].items())) or "nenhum"
    linhas += [
        "", "## Limpeza e LGPD", "",
        f"- blocos barrados no portao de qualidade da extracao: **{resultado['quarentena']}**",
        f"- dados pessoais mascarados: **{pii}**",
        f"- quase-duplicatas removidas: **{resultado['nd']}**",
        "", "## Corpus gerados", "",
        "| arquivo | exemplos | minimo do enunciado | situacao |", "|---|---|---|---|",
        f"| corpus_texto.jsonl | {resultado['texto']} | 50 | "
        f"{'ok' if resultado['texto'] >= 50 else 'abaixo'} |",
        f"| corpus_qa.jsonl | {resultado['qa']} | 100 | "
        f"{'ok' if resultado['qa'] >= 100 else 'abaixo'} |",
        f"| corpus_chat.jsonl | {resultado['chat']} | 30 | "
        f"{'ok' if resultado['chat'] >= 30 else 'abaixo'} |",
        "", f"Trechos de {TOKENS_MIN} a {TOKENS_MAX} tokens, contados com o tokenizer "
        f"de `{SLM_ALVO}`.",
    ]
    return "\n".join(linhas), resultado["arquivos"]


with gr.Blocks(title="Fabrica de Corpus CitizenTech") as demo:
    gr.Markdown(
        "# Fabrica de Corpus CitizenTech\n"
        "### Agricultura familiar: CAF, Pronaf, Garantia-Safra e assistencia tecnica\n"
        "Envie documentos oficiais em **PDF, TXT, DOCX ou HTML** (ou cole um texto) e a "
        "ferramenta devolve os tres corpus prontos para o fine-tuning de um SLM: texto puro, "
        "perguntas e respostas e conversacional.\n\n"
        "Pelo caminho ela barra paginas com extracao corrompida, remove cabecalho e rodape, "
        "junta palavra quebrada por hifen, mascara dado pessoal (preservando 0800, 135 e a "
        "caixa de e-mail de servico) "
        "e so deixa passar resposta que esta apoiada no texto de origem."
    )
    with gr.Row():
        with gr.Column():
            entrada_arquivos = gr.File(label="Documentos", file_count="multiple",
                                       file_types=[".pdf", ".txt", ".docx", ".html", ".htm"])
            entrada_texto = gr.Textbox(label="Ou cole um texto aqui", lines=6,
                                       placeholder="Ex.: o recado do sindicato sobre a adesao ao Garantia-Safra")
            botao = gr.Button("Gerar corpus", variant="primary")
        with gr.Column():
            saida_md = gr.Markdown(label="Relatorio")
            saida_arq = gr.File(label="Baixar os tres JSONL")

    botao.click(rodar_interface, [entrada_arquivos, entrada_texto], [saida_md, saida_arq])

# prevent_thread_lock deixa o notebook seguir rodando com a interface no ar.
# O share=True so funciona com internet liberada; se falhar, caimos no link local
# em vez de derrubar a execucao do notebook inteiro.
try:
    demo.launch(share=True, prevent_thread_lock=True, quiet=True)
    print("Interface no ar. Use o link publico acima para testar o upload.")
except Exception as e:
    print(f"(aviso) nao foi possivel criar o link publico ({type(e).__name__}).")
    demo.launch(share=False, prevent_thread_lock=True, quiet=True)
    print("Interface no ar apenas no endereco local.")


Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


Interface no ar. Use o link publico acima para testar o upload.


## 11. O que aprendemos

**A parte difícil foi extrair, não gerar.** Entramos achando que o trabalho seria montar os
pares. Encontramos cinco tipos de defeito de extração nos mesmos cinco documentos, e nenhum
deles aparece se a gente olhar só as primeiras páginas. O portão de qualidade foi a mudança
que mais melhorou o corpus, e nem estava no enunciado.

**Ler o documento rendeu mais do que inventar molde.** Nossa melhor família de perguntas
não usa molde: a cartilha do Pronaf já é escrita em pergunta e resposta, com as dúvidas do
agricultor, e passamos a extrair esse diálogo. Nenhum template que a gente escrevesse
chegaria a *"Aposentadoria também entra como renda da família?"*.

**Teste que sempre passa não ajuda.** O anonimizador parecia pronto e acertou 71% quando
testamos com variações de escrita que ele não previa. O mesmo vale para o verificador de
fidelidade: rodando só sobre resposta extraída, ele aprovava tudo. Só deu para saber se
funcionava depois de passar nele o texto de um LLM.

**Mascarar demais é tão ruim quanto mascarar de menos.** Uma regra ingênua de telefone
apagaria o 0800 do Disque-Intoxicação e o 135 do INSS. O NER trocou "PRONAF MULHER" por
`[NOME]`, ou seja, apagou uma linha de crédito inteira. Nos dois casos o corpus ficaria
mais limpo e o modelo, pior.

**O governo escreve para dois leitores no mesmo programa.** A cartilha do Pronaf fala com a
agricultora, em segunda pessoa e com exemplo do dia a dia. O manual do Garantia-Safra fala
com o gestor municipal, sobre indícios de desconformidade e cruzamento de bases. Os dois são
oficiais e tratam do mesmo assunto, e os dois entram no corpus. O risco é o modelo responder
à Dona Alzira em linguagem de auditoria.

**A dúvida mais importante da persona é a que as fontes menos respondem.** Fizemos uma
família para o "em que porta eu bato?" e ela rendeu 3 pares em 20 frases candidatas. Os
documentos citam sindicato e Emater bastante, mas quase nunca no formato "procure X para
fazer Y". O que falta aqui é material, não regra.

### O que faríamos com mais tempo

- Marcar cada trecho com o tipo de leitor (cidadão ou gestor) e usar isso como peso no
  fine-tuning, para o modelo aprender o conteúdo do manual e responder no tom da cartilha.
- Tratar a diagramação em caixas de texto pela posição das palavras na página. Hoje, quando
  o portão reprova uma página, o conteúdo bom dela vai junto.
- Afrouxar o verificador de fidelidade para aceitar paráfrase correta, comparando por
  embedding em vez de palavras em comum.
- Incluir material das Emateres estaduais, que tratam de cultura e região específicas e
  provavelmente respondem melhor o "em que porta eu bato".
- Fazer o fine-tuning e medir se o modelo respeita os limites que ensinamos, principalmente
  a recusa de indicar agrotóxico. Esta GS para antes disso, mas é o teste que fecharia o
  ciclo.